# SPI Results Visualization #

This code is comprised of multiple code segments that each create their own figures. The earlier sections include figures that focus on the ECR and economic costs (dollar value), that are not used in the final paper due to the focus on energetic efficiency. They are included here just for reader interest, and the more relevant figures are from the "Bio Analysis" section onward. Each  block of figure generation require the import of different Excel sheets' data (the corresponding pd.read block should be above the figure sections, with some blocks corresponding to multiple figures, hence some sections where there is one upload code block followed by multiple figures). If at any time the figures do not run properly, ensure the correct Excel data was read and that upload code block was rerun (to get a fresh version of the data).

In [1]:
import plotly
from plotly.subplots import make_subplots
import plotly.express as px
import plotly.graph_objs as go
import numpy as np
from scipy import stats
import pandas as pd
import sympy as sy
import math

from datetime import date, datetime

date = datetime.strftime(datetime.now(), '%Y-%m-%d')

In [2]:
#Global vars
#Change file locations depending on folder naming and ensure the below functions are run before figures are generated
inputDirectory_general = 'C://Users//austi//Documents//eCO2R TEA Code//ecor-tea//Final Repo//Visualization Docs////Input//'
outputDirectory_general = 'C://Users//austi//Documents//eCO2R TEA Code//ecor-tea//Final Repo//Visualization Docs////Output//'


In [3]:
def readInData(FileName, SheetName, Type):
    
    df = pd.read_excel(inputDirectory_general + FileName, sheet_name = SheetName)
    
    #Add Label Columns:
    if Type == 'ECR':
        print('Type= ECR')
        df = addColumn('Substrate', 'ECR_ProductGraph', df)
        df = addColumn('Substrate', 'SubstrateType', df)
        
    elif Type =='ECR-Sens':
        print('Type= ECR-Sens')
        df = addColumn('Substrate', 'ECR_ProductGraph', df)
        df = addColumn('Substrate', 'SubstrateType', df)
        df = addColumn('ParametersType', 'ParametersTypeGraph', df)
        df = addColumn('ParametersType', 'AnalysisType', df)
        
    elif Type == 'Bio':
        df = addColumn('Substrate', 'PathwayGraph', df)
        df = addColumn('Substrate', 'SubstrateType', df)        
    
    elif Type == 'ECR-Bio':
        df = addColumn('SubstratePathway', 'PathwayGraph', df)
        df = addColumn('Substrate', 'SubstrateType', df)
        
    return (df)

In [4]:
def addColumn(columnName, newColumn, df):

    if newColumn == 'SubstrateType':
        #Values for ECR Analysis:
        df.loc[df[columnName] == 'Acetaldehyde', newColumn] = 'C2'
        df.loc[df[columnName] == 'AceticAcid', newColumn] = 'C2'
        df.loc[df[columnName] == 'EthyleneGlycol', newColumn] = 'C2'
        df.loc[df[columnName] == 'Ethanol', newColumn] = 'C2'
        df.loc[df[columnName] == 'Glycolaldehyde', newColumn] = 'C2'
    
        df.loc[df[columnName] == 'Methanol', newColumn] = 'C1'
        df.loc[df[columnName] == 'FormicAcid', newColumn] = 'C1'
        df.loc[df[columnName] == 'CarbonMonoxide', newColumn] = 'C1'
    
        df.loc[df[columnName] == 'FormicAcid_0.5', newColumn] = 'C1+H2'
        df.loc[df[columnName] == 'FormicAcid_1', newColumn] = 'C1+H2'
        df.loc[df[columnName] == 'FormicAcid_2', newColumn] = 'C1+H2'
    
        df.loc[df[columnName] == 'CarbonMonoxide_0.5', newColumn] = 'C1+H2'
        df.loc[df[columnName] == 'CarbonMonoxide_1', newColumn] = 'C1+H2'
        df.loc[df[columnName] == 'CarbonMonoxide_2', newColumn] = 'C1+H2' 
        
        df.loc[df[columnName] == 'CarbonDioxide_2', newColumn] = 'C1+H2'
        df.loc[df[columnName] == 'CarbonDioxide_3', newColumn] = 'C1+H2'
        df.loc[df[columnName] == 'CarbonDioxide_4', newColumn] = 'C1+H2' 

        #Values for Bio Analysis:
        #df.loc[df[columnName] == 'Acetaldehyde', newColumn] = 'C2' #Same as for ECR Analysis
        df.loc[df[columnName] == 'Acetate', newColumn] = 'C2'
        #df.loc[df[columnName] == 'Ethanol', newColumn] = 'C2' #Same as for ECR Analysis
        df.loc[df[columnName] == 'EG_Glycolate', newColumn] = 'C2'
        df.loc[df[columnName] == 'EG_SACA', newColumn] = 'C2'
        df.loc[df[columnName] == 'Glycolaldehyde_Glycolate', newColumn] = 'C2'
        df.loc[df[columnName] == 'Glycolaldehyde_SACA', newColumn] = 'C2'
    
    
        df.loc[df[columnName] == 'Formate_Formolase', newColumn] = 'C1'
        df.loc[df[columnName] == 'Formate_RGP-Serine', newColumn] = 'C1'
        df.loc[df[columnName] == 'Formate_SerineCycle', newColumn] = 'C1'
        
        df.loc[df[columnName] == 'Methanol_Formolase', newColumn] = 'C1'
        df.loc[df[columnName] == 'Methanol_RuMP', newColumn] = 'C1'
        df.loc[df[columnName] == 'Methanol_SerineCycle', newColumn] = 'C1'
        df.loc[df[columnName] == 'Methanol_SerineCycle', newColumn] = 'C1'
        df.loc[df[columnName] == 'Methanol_SerineCycle', newColumn] = 'C1'
        df.loc[df[columnName] == 'Methanol_SerineCycle', newColumn] = 'C1'
        df.loc[df[columnName] == 'Methanol_MethylTransferase_1', newColumn] = 'C1'
        df.loc[df[columnName] == 'Methanol_MethylTransferase_0.5', newColumn] = 'C1'
        df.loc[df[columnName] == 'Methanol_MethylTransferase_0.1', newColumn] = 'C1'
        df.loc[df[columnName] == 'Methanol_MethylTransferase_0', newColumn] = 'C1'
    
        df.loc[df[columnName] == 'FOR_1:H2_0', newColumn] = 'C1+H2'
        df.loc[df[columnName] == 'FOR_1:H2_0.5', newColumn] = 'C1+H2'
        df.loc[df[columnName] == 'FOR_1:H2_1', newColumn] = 'C1+H2'
        df.loc[df[columnName] == 'FOR_1:H2_2', newColumn] = 'C1+H2'
    
        df.loc[df[columnName] == 'CO_1:H2_0', newColumn] = 'C1+H2'
        df.loc[df[columnName] == 'CO_1:H2_0.5', newColumn] = 'C1+H2'
        df.loc[df[columnName] == 'CO_1:H2_1', newColumn] = 'C1+H2'
        df.loc[df[columnName] == 'CO_1:H2_2', newColumn] = 'C1+H2'

        df.loc[df[columnName] == 'CO_2:H2_2', newColumn] = 'CO2+H2'
        
        df.loc[df[columnName] == 'Glucose', newColumn] = 'Sugar'
        df.loc[df[columnName] == 'Glycerol', newColumn] = 'Sugar'
        df.loc[df[columnName] == 'Xylose', newColumn] = 'Sugar'
               
    if newColumn == 'ECR_ProductGraph':   #For ECR only 
        #Add Substrate Name for Graphing
        df.loc[df[columnName] == 'Acetaldehyde', newColumn] = 'Acetaldehyde'
        df.loc[df[columnName] == 'AceticAcid', newColumn] = 'Acetic Acid'
        df.loc[df[columnName] == 'EthyleneGlycol', newColumn] = 'Ethylene Glycol'
        df.loc[df[columnName] == 'Ethanol', newColumn] = 'Ethanol'
        df.loc[df[columnName] == 'Glycolaldehyde', newColumn] = 'Glycolaldehyde'
    
    
        df.loc[df[columnName] == 'FormicAcid', newColumn] = 'Formic Acid'
        df.loc[df[columnName] == 'CarbonMonoxide', newColumn] = 'CO'
        df.loc[df[columnName] == 'Methanol', newColumn] = 'Methanol'

    
        df.loc[df[columnName] == 'FormicAcid_0.5', newColumn] = '1 FOR:0.5 H2'
        df.loc[df[columnName] == 'FormicAcid_1', newColumn] = '1 FOR:1 H2'
        df.loc[df[columnName] == 'FormicAcid_2', newColumn] = '1 FOR:2 H2'
    
        df.loc[df[columnName] == 'CarbonMonoxide_0.5', newColumn] = '1 CO:0.5 H2'
        df.loc[df[columnName] == 'CarbonMonoxide_1', newColumn] = '1 CO:1 H2'
        df.loc[df[columnName] == 'CarbonMonoxide_2', newColumn] = '1 CO:2 H2'

        df.loc[df[columnName] == 'CarbonDioxide_2', newColumn] = '1 CO2:2 H2'
        df.loc[df[columnName] == 'CarbonDioxide_3', newColumn] = '1 CO2:3 H2'
        df.loc[df[columnName] == 'CarbonDioxide_4', newColumn] = '1 CO2:4 H2' 
    
    if newColumn == 'PathwayGraph':
        df.loc[df[columnName] == 'Acetaldehyde', newColumn] = 'Acetaldehyde'
        df.loc[df[columnName] == 'Acetate', newColumn] = 'Acetic Acid'
        df.loc[df[columnName] == 'Ethanol', newColumn] = 'Ethanol'
        df.loc[df[columnName] == 'EG_Glycolate', newColumn] = 'EG - Glycolate'
        df.loc[df[columnName] == 'EG_SACA', newColumn] = 'EG - SACA'
        df.loc[df[columnName] == 'Glycolaldehyde_Glycolate', newColumn] = 'GlycAld - Glycolate'
        df.loc[df[columnName] == 'Glycolaldehyde_SACA', newColumn] = 'GlycAld - SACA'
    
    
        df.loc[df[columnName] == 'Formate_Formolase', newColumn] = 'Formate - Formolase'
        df.loc[df[columnName] == 'Formate_RGP-Serine', newColumn] = 'Formate - RGP'
        df.loc[df[columnName] == 'Formate_SerineCycle', newColumn] = 'Formate - Serine Cycle'
        
        df.loc[df[columnName] == 'Methanol_Formolase', newColumn] = 'Methanol - Formolase'
        df.loc[df[columnName] == 'Methanol_RuMP', newColumn] = 'Methanol - RuMP'
        df.loc[df[columnName] == 'Methanol_SerineCycle', newColumn] = 'Methanol - Serine Cycle'
        df.loc[df[columnName] == 'Methanol_MethylTransferase_1', newColumn] = 'Methanol Fixation - 1 ATP'
        df.loc[df[columnName] == 'Methanol_MethylTransferase_0.5', newColumn] = 'Methanol Fixation - 0.5 ATP'
        df.loc[df[columnName] == 'Methanol_MethylTransferase_0.1', newColumn] = 'Methanol Fixation - 0.1 ATP'
        df.loc[df[columnName] == 'Methanol_MethylTransferase_0', newColumn] = 'Methanol Fixation - no ATP'
    
        df.loc[df[columnName] == 'FOR_1:H2_0', newColumn] = '1 FOR:0 H2'
        df.loc[df[columnName] == 'FOR_1:H2_0.5', newColumn] = '1 FOR:0.5 H2'
        df.loc[df[columnName] == 'FOR_1:H2_1', newColumn] = '1 FOR:1 H2'
        df.loc[df[columnName] == 'FOR_1:H2_2', newColumn] = '1 FOR:2 H2'
    
        df.loc[df[columnName] == 'CO_1:H2_0', newColumn] = '1 CO:0 H2'
        df.loc[df[columnName] == 'CO_1:H2_0.5', newColumn] = '1 CO:0.5 H2'
        df.loc[df[columnName] == 'CO_1:H2_1', newColumn] = '1 CO:1 H2'
        df.loc[df[columnName] == 'CO_1:H2_2', newColumn] = '1 CO:2 H2'

        df.loc[df[columnName] == 'CO_2:H2_2', newColumn] = '1 CO2:X H2'
        
        df.loc[df[columnName] == 'Glucose', newColumn] = 'Glucose'
        df.loc[df[columnName] == 'Glycerol', newColumn] = 'Glycerol'
        df.loc[df[columnName] == 'Xylose', newColumn] = 'Xylose'
        
        
    if newColumn == 'ParametersTypeGraph': #Senstivity Analysis only
        
        df.loc[df[columnName] == 'Worst', newColumn] = 'Worst'
        df.loc[df[columnName] == 'Base', newColumn] = 'Base'
        df.loc[df[columnName] == 'Best', newColumn] = 'Best'
        
        df.loc[df[columnName] == 'ElectricityPrice_Worst', newColumn] = 'Electricity Price'
        df.loc[df[columnName] == 'ElectricityPrice_Best', newColumn] = 'Electricity Price'
        
        df.loc[df[columnName] == 'CO2Price_Worst', newColumn] = 'CO2 Price'
        df.loc[df[columnName] == 'CO2Price_Best', newColumn] = 'CO2 Price'
        
        df.loc[df[columnName] == 'CarbonTax_Worst', newColumn] = 'Carbon Tax'
        df.loc[df[columnName] == 'CarbonTax_Best', newColumn] = 'Carbon Tax'
        
        df.loc[df[columnName] == 'Conversion_Worst', newColumn] = 'CO2 Conversion'
        df.loc[df[columnName] == 'Conversion_Best', newColumn] = 'CO2 Conversion'
    
        df.loc[df[columnName] == 'ElectrolyzerPrice_Worst', newColumn] = 'Electrolyzer Price'
        df.loc[df[columnName] == 'ElectrolyzerPrice_Best', newColumn] = 'Electrolyzer Price'
        
        df.loc[df[columnName] == 'FaradaicEfficiency_Worst', newColumn] = 'Faradaic Efficiency'
        df.loc[df[columnName] == 'FaradaicEfficiency_Best', newColumn] = 'Faradaic Efficiency'

        df.loc[df[columnName] == 'CurrentDensity_Worst', newColumn] = 'Current Density'
        df.loc[df[columnName] == 'CurrentDensity_Best', newColumn] = 'Current Density'
        
        df.loc[df[columnName] == 'CellVoltage_Worst', newColumn] = 'Cell Voltage'
        df.loc[df[columnName] == 'CellVoltage_Best', newColumn] = 'Cell Voltage'
        
    if newColumn == 'AnalysisType': #Senstivity Analysis only
        
        df.loc[df[columnName] == 'Worst', newColumn] = 'Worst-OVERALL'
        df.loc[df[columnName] == 'Base', newColumn] = 'Base'
        df.loc[df[columnName] == 'Best', newColumn] = 'Best-OVERALL'
        
        df.loc[df[columnName] == 'ElectricityPrice_Worst', newColumn] = 'Worst'
        df.loc[df[columnName] == 'ElectricityPrice_Best', newColumn] = 'Best'
        
        df.loc[df[columnName] == 'CO2Price_Worst', newColumn] = 'Worst'
        df.loc[df[columnName] == 'CO2Price_Best', newColumn] = 'Best'
        
        df.loc[df[columnName] == 'CarbonTax_Worst', newColumn] = 'Worst'
        df.loc[df[columnName] == 'CarbonTax_Best', newColumn] = 'Best'
        
        df.loc[df[columnName] == 'Conversion_Worst', newColumn] = 'Worst'
        df.loc[df[columnName] == 'Conversion_Best', newColumn] = 'Best'
    
        df.loc[df[columnName] == 'ElectrolyzerPrice_Worst', newColumn] = 'Worst'
        df.loc[df[columnName] == 'ElectrolyzerPrice_Best', newColumn] = 'Best'
        
        df.loc[df[columnName] == 'FaradaicEfficiency_Worst', newColumn] = 'Worst'
        df.loc[df[columnName] == 'FaradaicEfficiency_Best', newColumn] = 'Best'

        df.loc[df[columnName] == 'CurrentDensity_Worst', newColumn] = 'Worst'
        df.loc[df[columnName] == 'CurrentDensity_Best', newColumn] = 'Best'
        
        df.loc[df[columnName] == 'CellVoltage_Worst', newColumn] = 'Worst'
        df.loc[df[columnName] == 'CellVoltage_Best', newColumn] = 'Best'   
    
    return(df)

## ECR Analysis ##

**Notes:**
- ECR costs comparison
- Two scenarios: Ideal or Realistic
- Three cases: Worst, Base, Best

In [5]:
FileName = '2023-12-14_Type-ECR_Case-Category_1_Analysis-Categories_FULL.xlsx'
SheetName = 'Overall-ECR'
Type = 'ECR'

df = readInData(FileName, SheetName, Type)
display(df) #Check

Type= ECR


,Unnamed: 0,Substrate,Formula_S,C_Sn,H_Sn,O_Sn,MW_S,N_electrons,ParametersType,CellVoltage,...,Cost_elec_mol,Cost_elec_mass,Cost_CO2_mass,Cost_electrolyzer_mass,CT_savings_mass,ProductionCost_massNOCT,ProductionCost_mass,CO2_ECons_mass,ECR_ProductGraph,SubstrateType
0,0,Acetaldehyde,C2H4O,2,4,1,44.05176,10,Categories-Category_1,2.0,...,0.029035,659.109605,825.883007,565.808985,905.807169,2050.801596,1144.994428,5.328277,Acetaldehyde,C2
1,1,AceticAcid,CH3COOH,2,4,2,60.05076,8,Categories-Category_1,4.0,...,0.039819,663.094751,1054.823691,18.974334,462.761361,1736.892775,1274.131414,2.722126,Acetic Acid,C2
2,2,Ethanol,C2H5OH,2,6,1,46.06744,12,Categories-Category_1,2.5,...,0.052263,1134.486497,1184.619766,77.911491,1299.260389,2397.017755,1097.757366,7.642708,Ethanol,C2
3,3,EthyleneGlycol,(CH2OH)2,2,6,2,62.06644,10,Categories-Category_1,2.0,...,0.029035,467.804149,586.171851,401.583877,642.898159,1455.559876,812.661718,3.781754,Ethylene Glycol,C2
4,4,FormicAcid,CHOOH,1,1,2,45.01654,2,Categories-Category_1,4.0,...,0.009955,221.137296,541.193938,6.327803,237.427018,768.659037,531.232019,1.396630,Formic Acid,C1
5,5,Glycolaldehyde,C2H4O2,2,4,2,60.05076,8,Categories-Category_1,4.0,...,0.557471,9283.326511,8860.519001,3984.610075,9717.988582,22128.455586,12410.467005,57.164639,Glycolaldehyde,C2
6,6,Methanol,CH3OH,1,4,1,32.04106,6,Categories-Category_1,2.5,...,0.026131,815.561168,798.375974,56.009117,875.638166,1669.946260,794.308094,5.150813,Methanol,C1
7,7,CarbonMonoxide,CO,1,0,1,28.00970,2,Categories-Category_1,4.0,...,0.009955,355.406732,869.794342,10.169898,381.587195,1235.370971,853.783776,2.244631,CO,C1
8,8,CarbonMonoxide_0.5,CO,1,0,1,28.00970,2,Categories-Category_1,4.0,...,0.010453,360.189035,587.665448,10.306743,257.814519,958.161225,700.346706,1.571241,1 CO:0.5 H2,C1+H2
9,9,CarbonMonoxide_1,CO,1,0,1,28.00970,2,Categories-Category_1,4.0,...,0.013937,464.099552,567.900279,13.280123,249.143348,1045.279953,796.136605,1.571241,1 CO:1 H2,C1+H2


In [38]:
scenario = 'Categories' #Ideal or Realistic
case = 'Category_2' #Worst, Base, Best
FileName = '2023-12-11_Type-Bio_Case-'+ case +'_Analysis-' + scenario + '_FULL.xlsx'
SheetName = 'Overall-ECR-Bio'
Type = 'ECR-Bio'

df = readInData(FileName, SheetName, Type)

df.loc[df['SubstratePathway'] == 'FOR_1:H2_0', 'SubstrateType'] = 'C1+H2'
df.loc[df['SubstratePathway'] == 'CO_1:H2_0', 'SubstrateType'] = 'C1+H2'
display(df)

,Unnamed: 0,Substrate,Formula_S,C_Sn,H_Sn,O_Sn,MW_S,N_electrons,ParametersType,CellVoltage,...,YieldBP_Molar,YieldBP_Mass,Bioconversion,BioproductionCost_massNOCT,BioproductionCost_mass1,CO2_OverallCons_mass,CT_OverallSavings_mass,BioproductionCost_mass2,PathwayGraph,SubstrateType
0,0,Acetaldehyde,C2H4O,2,4,1,44.05176,10,Categories-Category_2,1.176,...,0.437381,0.894775,100,481.937538,102.314035,1.953413,332.080178,149.857359,Acetaldehyde,C2
1,1,Acetaldehyde,C2H4O,2,4,1,44.05176,10,Categories-Category_2,1.176,...,0.444821,0.909995,100,474.448990,101.174637,1.953413,332.080178,142.368812,Acetaldehyde,C2
2,2,Acetaldehyde,C2H4O,2,4,1,44.05176,10,Categories-Category_2,1.176,...,0.496486,0.858133,100,501.056346,105.223001,1.734052,294.788860,206.267486,Acetaldehyde,C2
3,3,Acetaldehyde,C2H4O,2,4,1,44.05176,10,Categories-Category_2,1.176,...,0.510882,0.882473,100,488.179538,103.263768,1.735118,294.970012,193.209527,Acetaldehyde,C2
4,4,Acetaldehyde,C2H4O,2,4,1,44.05176,10,Categories-Category_2,1.176,...,0.403953,0.826391,100,518.989156,107.951514,1.953413,332.080178,186.908977,Acetaldehyde,C2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1344,1344,Methanol,CH3OH,1,4,1,32.04106,6,Categories-Category_2,1.213,...,0.049362,0.211188,100,1400.842046,571.593321,4.877934,829.248725,571.593321,Methanogenesis - no ATP,C1
1345,1345,Methanol,CH3OH,1,4,1,32.04106,6,Categories-Category_2,1.213,...,0.154667,0.677048,100,460.481491,201.818141,1.521549,258.663350,201.818141,Methanogenesis - no ATP,C1
1346,1346,Methanol,CH3OH,1,4,1,32.04106,6,Categories-Category_2,1.213,...,0.178462,0.684940,100,455.570197,199.886887,1.504019,255.683310,199.886887,Methanogenesis - no ATP,C1
1347,1347,Methanol,CH3OH,1,4,1,32.04106,6,Categories-Category_2,1.213,...,0.122105,0.304705,100,981.403486,406.658745,3.380851,574.744741,406.658745,Methanogenesis - no ATP,C1


In [39]:
#Add Glucose Cost Data and make new DF
df2 = pd.read_excel(inputDirectory_general + '2021-08-29_HybridCosts-GlucoseANDConventionalCosts.xlsx', sheet_name = 'Data')


if case is 'Category_2':
    costColumn = 'Cost_Best'
elif case is 'Category_1':
    costColumn = 'Cost_Base'
    
    
df2 = df2[['SubstratePathway', costColumn]]
df2 = df2.rename(columns = {'SubstratePathway': 'PathwayGraph', costColumn:'BioproductionCost_mass2'})
df2['SubstrateType'] = 'Sugar'
#display(df2)

df3 = df[['PathwayGraph', 'BioproductionCost_mass2', 'SubstrateType']]
df3 = df3.append(df2, ignore_index=True)
#display(df3)

df = df3
display(df)

<>:5: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:7: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:5: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:7: SyntaxWarning:

"is" with a literal. Did you mean "=="?

C:\Users\austi\AppData\Local\Temp\ipykernel_7752\3008018299.py:5: SyntaxWarning:

"is" with a literal. Did you mean "=="?

C:\Users\austi\AppData\Local\Temp\ipykernel_7752\3008018299.py:7: SyntaxWarning:

"is" with a literal. Did you mean "=="?

C:\Users\austi\AppData\Local\Temp\ipykernel_7752\3008018299.py:17: FutureWarning:

The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.



,PathwayGraph,BioproductionCost_mass2,SubstrateType
0,Acetaldehyde,149.857359,C2
1,Acetaldehyde,142.368812,C2
2,Acetaldehyde,206.267486,C2
3,Acetaldehyde,193.209527,C2
4,Acetaldehyde,186.908977,C2
...,...,...,...
1401,Glucose,1047.863777,Sugar
1402,Glucose,1360.613061,Sugar
1403,Glucose,259.452821,Sugar
1404,Glucose,777.039974,Sugar


### ECR-Bio Box Plot ###

In [40]:
ORI = 'h' #'v' for vertical, 'h' for horizontal

if ORI is 'v':
    X = 'PathwayGraph'
    Y = 'BioproductionCost_mass2'
    W = 900
    H = 500
    
elif ORI is 'h':
    X = 'BioproductionCost_mass2'
    Y = 'PathwayGraph'
    W = 900
    H = 800

    
layout = dict(xaxis = dict(title = 'Substrate test', showgrid=False, ticks='inside'))

fig = px.box(df, x=X, y=Y,
             orientation=ORI,
             color = 'SubstrateType', color_discrete_map={'C2': "Red", 'C1': 'Blue', 'C1+H2': 'Green'},
             title = str('Cost of Bioproduction (ECR Costs: ' + scenario + '-' + case +')'),
             points = 'outliers',
             category_orders={'PathwayGraph':['Acetaldehyde', 'Acetic Acid', 'Ethanol', 'EG - Glycolate', 'EG - SACA',
             'GlycAld - Glycolate', 'GlycAld - SACA', 'Formate - Formolase', 'Formate - RGP', 'Formate - Serine Cycle', 
             'Methanol - Formolase', 'Methanol - RuMP', 'Methanol - Serine Cycle', 
             '1 CO:0 H2', '1 CO:0.5 H2', '1 CO:1 H2', '1 CO:2 H2', 
             '1 FOR:0 H2', '1 FOR:0.5 H2', '1 FOR:1 H2', '1 FOR:2 H2'],
             'SubstrateType': ['C2', 'C1', 'C1+H2']}
             #range_x =[0,14]
            )


fig.update_layout(template="simple_white", plot_bgcolor = "white", title = {'x': 0.52},
                 width=W, height=H, title_font_size=16, legend_font_size=16, legend_title_font_size=16,
                legend_title_text='Substrate Type'
                 )

#The following is needed to change the width of the boxes, and ensure they're aligned with the labels
fig.update_traces(width=0.25)


#Horizontal vs Vertical box plot need opposite axis settings:
if ORI is 'v':
    fig.update_xaxes(title_text='Substrate', title_font_size=16,
                     visible=True, showline=True, linewidth=1, color='black', mirror = True,
                     tickfont_size=14,
                     type='category'#, tickson='boundaries'
                     )

    fig.update_yaxes(title_text='Cost of Bioproduction ($CA/tonne)', title_font_size=18,
                    visible=True, showline=True, linewidth=1, color='black', mirror = True,
                    tickmode = 'linear', tick0 = 0, dtick= 1000, tickfont_size=14,
                    showgrid=True, gridcolor='#eee'
                    )
    
    
elif ORI is 'h':
    
    fig.update_yaxes(title_text='Substrate', title_font_size=16,
                 visible=True, showline=True, linewidth=1, color='black', mirror = True,
                 tickfont_size=16,
                 type='category'#, tickson='boundaries'
                 )
                 #tickson="boundaries")

    fig.update_xaxes(title_text='Cost of Bioproduction ($CA/tonne)', title_font_size=16,
                visible=True, showline=True, linewidth=1, color='black', mirror = True,
                #tickmode = 'linear', 
                tickfont_size=16, #tick0 = 0, dtick= np.log10(1),
                showgrid=True, gridcolor='#eee', type='log',
                )
    
    if case is 'Best':
        fig.update_xaxes(tickvals=np.arange(20, 120, 20).tolist() + np.arange(200, 1200, 200).tolist() + np.arange(2000, 12000, 2000).tolist(),
                ticktext=["2","4","6","8","100","2","4","6","8","1k", "2","4","6","8","10k"], tickangle=-45
                )

#fig.update_traces(texttemplate='%{text:.2s}', textposition='outside')
fig.update_layout(legend=dict(yanchor="bottom", y=0.01, xanchor="right", x=0.99))
    
    #category_orders={'Substrate':['C2', 'C1', 'C1+H2']}
#color_discrete_sequence=px.colors.qualitative.Set1,
#"C1":['CarbonMonoxide','FormicAcid','Methanol']
#for t in fig.data:
   # t.marker.line.width = 1
    #t.marker.line.color = 'black'


fig.show()
fig.write_image(outputDirectory_general + date + "_ECR-Bio_scenario-" + scenario + "_case-" + case + ".png", scale=5)   

<>:3: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:9: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:43: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:57: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:73: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:3: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:9: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:43: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:57: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:73: SyntaxWarning:

"is" with a literal. Did you mean "=="?

C:\Users\austi\AppData\Local\Temp\ipykernel_7752\3679041780.py:3: SyntaxWarning:

"is" with a literal. Did you mean "=="?

C:\Users\austi\AppData\Local\Temp\ipykernel_7752\3679041780.py:9: SyntaxWarning:

"is" with a literal. Did you mean "=="?

C:\Users\austi\AppData\Local\Temp\ipykernel_7752\3679041780.py:43: SyntaxWarning:

"is" with a literal. Did you mean "

### ECR-Bio Scatterplot ###

In [5]:
scenario = 'Categories' #Ideal or Realistic
case = 'Category_2' #Worst, Base, Best
FileName = '2024-05-08_Type-Bio_Case-'+ case +'_Analysis-' + scenario + '_FULL.xlsx'
SheetName = 'Overall-ECR-Bio'
Type = 'ECR-Bio'

df = readInData(FileName, SheetName, Type)
n = 57

df.loc[df['SubstratePathway'] == 'FOR_1:H2_0', 'SubstrateType'] = 'C1+H2'
df.loc[df['SubstratePathway'] == 'CO_1:H2_0', 'SubstrateType'] = 'C1+H2'


#To graph only a subset of bioproducts:
#df = df[(df['BioProduct'].isin(['Propane-PW1','Butanol-PW2', '1-3-Diaminopropane', 'Isoprene', '2-3-Butanediol','Lysine_L', 'Catechol', 'Adipate', 'Succinate', 'Malonate']))]
#n = 10

display(df)


,Unnamed: 0,Substrate,Formula_S,C_Sn,H_Sn,O_Sn,MW_S,N_electrons,ParametersType,CellVoltage,...,CCRec_ECR,Flue_gas_rec,Bioreactor_rec,BioRec_COInput,BioRec_COBio,BioRec_Rec,BioRec_ECR,BioproductionCost_mass2_alt,PathwayGraph,SubstrateType
0,562.0,CarbonMonoxide,CO,1.0,0.0,1.0,28.00970,2.0,Categories-Category_2,1.333,...,17.016689,24.010994,19.015062,1.998373,0.0,0.000000,17.016689,1.100000,NaN,C1
1,337.0,Methanol,CH3OH,1.0,4.0,1.0,32.04106,6.0,Categories-Category_2,1.213,...,29.656178,35.570239,31.345910,1.689732,0.0,0.000000,29.656178,203.475151,Methanol Fixation - no ATP,C1
2,281.0,Methanol,CH3OH,1.0,4.0,1.0,32.04106,6.0,Categories-Category_2,1.213,...,29.656178,35.570239,31.345910,1.689732,0.0,0.000000,29.656178,203.475151,Methanol Fixation - 0.1 ATP,C1
3,1039.0,CarbonMonoxide_1,CO,1.0,0.0,1.0,28.00970,2.0,Categories-Category_2,1.333,...,26.089033,35.137348,28.674266,2.585233,0.0,0.000000,26.089033,45.468681,1 CO:1 H2,C1+H2
4,1040.0,CarbonMonoxide_1,CO,1.0,0.0,1.0,28.00970,2.0,Categories-Category_2,1.333,...,26.204090,35.292311,28.800725,2.596635,0.0,0.000000,26.204090,45.528573,1 CO:1 H2,C1+H2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1556,1506.0,FormicAcid,CHOOH,1.0,1.0,2.0,45.01654,2.0,Categories-Category_2,1.400,...,323.123307,544.716817,532.760478,4.782536,0.0,204.854635,323.123307,7087.031293,Formate - Serine Cycle,C1
1557,1507.0,FormicAcid,CHOOH,1.0,1.0,2.0,45.01654,2.0,Categories-Category_2,1.400,...,330.759971,557.590599,545.619562,4.788415,0.0,210.071176,330.759971,7322.859925,Formate - Serine Cycle,C1
1558,1508.0,FormicAcid,CHOOH,1.0,1.0,2.0,45.01654,2.0,Categories-Category_2,1.400,...,334.494022,563.885410,551.932416,4.781198,0.0,212.657196,334.494022,7413.999847,Formate - Serine Cycle,C1
1559,1509.0,FormicAcid,CHOOH,1.0,1.0,2.0,45.01654,2.0,Categories-Category_2,1.400,...,346.902147,584.802858,572.811479,4.796551,0.0,221.112781,346.902147,7710.788470,Formate - Serine Cycle,C1


In [6]:
#df2 = readInData('2021-08-21_HybridCosts-GlucoseANDConventionalCosts.xlsx', 'Data', 'ECR-Bio')
df2 = pd.read_excel(inputDirectory_general + '2021-08-29_HybridCosts-GlucoseANDConventionalCosts.xlsx', sheet_name = 'Data')

#To graph only a subset of bioproducts:
#df2 = df2[(df2['BioProduct'].isin(['Propane-PW1','Propane-PW2', 'Butane', 'Pentane', 'Hexane','Heptane', 'Octane', 'Nonane']))]
#df2 = df2[(df2['BioProduct'].isin(['Propane-PW1','Butanol-PW2', '1-3-Diaminopropane', 'Isoprene', '2-3-Butanediol','Lysine_L', 'Catechol', 'Adipate', 'Succinate', 'Malonate']))]
display(df2)

,SubstratePathway,BioProduct,BioProduct_Graph,Formula_BP,MW_BP,Deg_RedBP,BoilP_degC,CO2_Flux,Obj_Flux,Sep_Cost_Base,MassYield,"X_CO2,BP","X_CO2,HY80","X_CO2,HY90",Cost_Base,Cost_Best,Conventional Cost,Unnamed: 17,Unnamed: 18
0,Glucose,1-3-Butanediol-PW1,"1,3-Butanediol (PW1)",C4H10O2,90.11920,5.500000,207,16.363636,10.909091,26.093245,0.545715,0.732530,2.634611,2.260484,528.796697,460.684187,NaN,NaN,NaN
1,Glucose,1-3-Butanediol-PW2,"1,3-Butanediol (PW2)",C4H10O2,90.11920,5.500000,207,16.363636,10.909091,26.093245,0.545715,0.732530,2.634611,2.260484,528.796697,460.684187,NaN,NaN,NaN
2,Glucose,1-3-Diaminopropane,"1,3-Diaminopropane",C3H12N2,76.13958,6.000000,139.3,18.802597,13.732468,26.093245,0.580389,0.791426,2.374553,2.022778,516.215998,450.613822,NaN,NaN,NaN
3,Glucose,1-3-Propanediol,"1,3-Propanediol",C3H8O2,76.09282,5.333333,213,16.057778,14.647407,26.093245,0.618678,0.634063,2.335980,2.005975,467.458111,404.301070,NaN,NaN,NaN
4,Glucose,1-4-Butanediol,"1,4-Butanediol",C4H10O2,90.11920,5.500000,230,16.865116,10.783721,26.093245,0.539444,0.763756,2.642530,2.264054,538.501770,469.900750,NaN,NaN,NaN
5,Glucose,1-Propanol-PW1,1-Propanol (PW1),C3H8O,60.09382,6.000000,97,21.455814,12.848062,26.093245,0.428576,1.223007,3.064449,2.588065,715.541151,635.943825,NaN,NaN,NaN
6,Glucose,1-Propanol-PW2,1-Propanol (PW2),C3H8O,60.09382,6.000000,97,24.304455,11.898515,26.093245,0.396902,1.495942,3.133670,2.619268,800.368552,716.501395,NaN,NaN,NaN
7,Glucose,2-3-Butanediol,"2,3-Butanediol",C4H10O2,90.11920,5.500000,177,16.363636,10.909091,26.093245,0.545715,0.732530,2.634611,2.260484,528.796697,460.684187,NaN,NaN,NaN
8,Glucose,2-Methyl-1-Butanol,2-Methyl-1-Butanol,C5H12O,88.14658,6.000000,129,21.964029,7.607194,26.093245,0.372212,1.441562,3.495142,2.946620,825.614558,737.915120,NaN,NaN,NaN
9,Glucose,2-Pyrone-4-6-Dicarboxylate,"2-Pyrone-4,6-Dicarboxylate",C7H2O6,182.08458,2.571429,355.5,-2.473866,8.924838,26.093245,0.902057,-0.066997,2.104009,1.877674,243.486572,191.973018,NaN,NaN,NaN


In [16]:
##----- Bioproduction Yields Scatterplot -----##

#df = df.loc[df['MoleculeType_P'] == 'Hydrocarbon (alkane)']
#df = df.sort_values(by='C_P', ascending=0)
df = df.sort_values(by='Deg_RedBP', ascending=0)
#display(df)

ORI = 'h' #'v' for vertical, 'h' for horizontal
S = 'SubstratePathway'
maximum = 7000 #Above $6K it's only the Formate-Serine cycle and Formate-RGP pathways
minimum = -2000

if ORI is 'v':
    X = 'BioProduct_Graph'
    Y = 'BioproductionCost_mass2_alt' #'BioproductionCost_massNOCT'
    W = 1400
    H = 800
    title_pos = 0.45
    #For glucose costs:
    X2 = X
    if case is 'Category_2':
        Y2 = 'Cost_Best'
    elif case is 'Category_1':
        Y2 = 'Cost_Base'
    
elif ORI is 'h':
    X = 'BioproductionCost_mass2_alt'
    Y = 'BioProduct_Graph'
    W = 1200 #1000
    H = 1500 #600
    title_pos = 0.6
    #For glucose costs:
    Y2 = Y
    if case is 'Category_2':
        X2 = 'Cost_Best'
    elif case is 'Category_1':
        X2 = 'Cost_Base' 


markerSize=7

fig = go.Figure()


dfS1 = df2.loc[df2[S] == 'Glucose']
dataS1 = go.Scatter(x=dfS1[X2], y=dfS1[Y2], mode='markers', orientation=ORI,
                    marker_color= 'gold', marker_line_color='black', marker_line_width=0.75, marker_symbol= 'diamond', marker_size=markerSize,
                    name = 'Glucose'
                    )

dfS2 = df.loc[df[S] == 'Glycerol']
dataS2 = go.Scatter(x=dfS2[X], y=dfS2[Y], mode='markers', orientation=ORI,
                    marker_color= 'red',marker_symbol= 'diamond', marker_size=markerSize,
                    name = 'Glycerol'
                    )

dfS3 = df.loc[df[S] == 'Xylose'] 
dataS3 = go.Scatter(x=dfS3[X], y=dfS3[Y], mode='markers', orientation=ORI,
                    marker_color= 'blue', marker_symbol= 'diamond', marker_size=markerSize,
                    name = 'Xylose'
                    )

#ECR 1-C compounds; iJO1366
dfC11 = df.loc[df[S] == 'Methanol_Formolase']
dataC11 = go.Scatter(x=dfC11[X], y=dfC11[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkviolet', marker_symbol= 'square', marker_size = markerSize+1,
                    name = 'MeOH - Formolase',    
                    )

dfC12 = df.loc[df[S] == 'Methanol_RuMP']
dataC12 = go.Scatter(x=dfC12[X], y=dfC12[Y], mode='markers', orientation=ORI,
                    marker_color= 'deepskyblue', marker_symbol= 'square', marker_size = markerSize-1,
                    name = 'MeOH - RuMP',
                    )

dfC13 = df.loc[df[S] == 'Methanol_SerineCycle']
dataC13 = go.Scatter(x=dfC13[X], y=dfC13[Y], mode='markers', orientation=ORI,
                    marker_color= 'red', marker_symbol= 'square', marker_size = markerSize,
                    name = 'MeOH - SerineCycle'    
                    )

dfC14 = df.loc[df[S] == 'Formate_Formolase']
dataC14 = go.Scatter(x=dfC14[X], y=dfC14[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkviolet', marker_symbol= 'x', marker_size = markerSize,
                    name = 'Formate - Formolase',
                    )

dfC15 = df.loc[df[S] == 'Formate_RGP-Serine']
dataC15 = go.Scatter(x=dfC15[X], y=dfC15[Y],mode='markers', orientation=ORI,
                    marker_color= 'olivedrab', marker_symbol= 'x', marker_size = markerSize,
                    name = 'Formate - RGP',
                    )

dfC16 = df.loc[df[S] == 'Formate_SerineCycle']
dataC16 = go.Scatter(x=dfC16[X], y=dfC16[Y],mode='markers', orientation=ORI,
                    marker_color= 'red', marker_symbol= 'x', marker_size = markerSize,
                    name = 'Formate - Serine Cycle',
                    )

#ECR 2-C compounds; iJO1366
dfC21 = df.loc[df[S] == 'Acetaldehyde']
dataC21 = go.Scatter(x=dfC21[X], y=dfC21[Y], mode='markers', orientation=ORI,
                    marker_color='darkblue', marker_symbol= 'circle', marker_size =markerSize,
                    name = 'Acetaldehyde',
                    )

dfC22 = df.loc[df[S] == 'Acetate']
dataC22 = go.Scatter(x=dfC22[X], y=dfC22[Y], mode='markers', orientation=ORI,
                    marker_color= 'red', marker_symbol= 'circle', marker_size =markerSize,
                    name = 'Acetate',
                    )


dfC23 = df.loc[df[S] == 'Ethanol']
dataC23 = go.Scatter(x=dfC23[X], y=dfC23[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkorange',marker_symbol= 'circle', marker_size =markerSize,
                    name = 'Ethanol',
                    )

dfC24 = df.loc[df[S] == 'EG_Glycolate']
dataC24 = go.Scatter(x=dfC24[X], y=dfC24[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkblue', marker_symbol= 'triangle-up', marker_size =markerSize,
                    name = 'EG - Glycolate',
                    )

dfC25 = df.loc[df[S] == 'EG_SACA']
dataC25 = go.Scatter(x=dfC25[X], y=dfC25[Y],mode='markers', orientation=ORI,
                    marker_color= 'red', marker_symbol= 'triangle-up', marker_size =markerSize,
                    name = 'EG - SACA',
                    )



dfC26 = df.loc[df[S] == 'Glycolaldehyde_Glycolate']
dataC26 = go.Scatter(x=dfC26[X], y=dfC26[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkorange', marker_symbol= 'triangle-up', marker_size =markerSize,
                    name = 'GlyAld - Glycolate',
                    )

dfC27 = df.loc[df[S] == 'Glycolaldehyde_SACA']
dataC27 = go.Scatter(x=dfC27[X], y=dfC27[Y], mode='markers', orientation=ORI,
                    marker_color= 'deepskyblue', marker_symbol= 'triangle-up', marker_size =markerSize,
                    name = 'GlyAld - SACA',
                    )
#ECR 3-C compounds; iJO1366
dfC31 = df.loc[df[S] == 'N_Propanol']
dataC31 = go.Scatter(x=dfC31[X], y=dfC31[Y], mode='markers', orientation=ORI,
                    marker_color= 'black', marker_symbol= 'circle', marker_size =markerSize,
                    name = 'n-Propanol',
                    )

#ECR - iHN637 - CO + Formate Mixes
dfH1 = df.loc[df[S] == 'FOR_1:H2_0'] 
dataH1 = go.Scatter(x=dfH1[X], y=dfH1[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkgreen', marker_symbol= 'star-square-open', marker_size =markerSize,
                    name = '1 FOR:0 H2',
                    )

dfH2 = df.loc[df[S] == 'FOR_1:H2_0.5']
dataH2 = go.Scatter(x=dfH2[X], y=dfH2[Y], mode='markers', orientation=ORI,
                    marker_color= 'red', marker_symbol= 'star-square-open', marker_size =markerSize,
                    name = '1 FOR:0.5 H2',
                    )

dfH3 = df.loc[df[S] == 'FOR_1:H2_1']
dataH3 = go.Scatter(x=dfH3[X], y=dfH3[Y], mode='markers', orientation=ORI,
                    marker_color= 'dodgerblue', marker_symbol= 'star-square-open', marker_size =markerSize,
                    name = '1 FOR:1 H2',
                    )

dfH4 = df.loc[df[S] == 'FOR_1:H2_2']
dataH4 = go.Scatter(x=dfH4[X], y=dfH4[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkviolet', marker_symbol= 'star-square-open', marker_size =markerSize,
                    name = '1 FOR:2 H2',
                    )

dfH5 = df.loc[df[S] == 'CO_1:H2_0']
dataH5 = go.Scatter(x=dfH5[X], y=dfH5[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkgreen', marker_symbol= 'diamond-tall-open', marker_size =markerSize+0.5,
                    name = '1 CO:0 H2',
                    )

dfH6 = df.loc[df[S] == 'CO_1:H2_0.5'] 
dataH6 = go.Scatter(x=dfH6[X], y=dfH6[Y], mode='markers', orientation=ORI,
                    marker_color= 'red', marker_symbol= 'diamond-tall-open', marker_size =markerSize+0.5,
                    name = '1 CO:0.5 H2',
                    )

dfH7 = df.loc[df[S] == 'CO_1:H2_1']
dataH7 = go.Scatter(x=dfH7[X], y=dfH7[Y], mode='markers', orientation=ORI,
                    marker_color= 'dodgerblue', marker_symbol= 'diamond-tall-open', marker_size =markerSize+0.5,
                    name = '1 CO:1 H2',
                    )

dfH8 = df.loc[df[S] == 'CO_1:H2_2']
dataH8 = go.Scatter(x=dfH8[X], y=dfH8[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkviolet', marker_symbol= 'diamond-tall-open',  marker_size =markerSize+0.5,
                    name = '1 CO:2 H2',
                    )

dfH9 = df.loc[df[S] == 'CO_2:H2_3']
dataH9 = go.Scatter(x=dfH9[X], y=dfH9[Y], mode='markers', orientation=ORI,
                    marker_color= 'black', marker_symbol= 'circle-open',  marker_size =markerSize,
                    name = '1 CO2:X H2',
                    )

dfH14 = df.loc[df[S] == 'CO_2:CO1_4']
dataH14 = go.Scatter(x=dfH14[X], y=dfH14[Y], mode='markers', orientation=ORI,
                    marker_color= 'black', marker_symbol= 'diamond-tall-open',  marker_size =markerSize,
                    name = 'Pure CO with Allowable Emissions',
                    )

dfH15 = df.loc[df[S] == 'Flue_Gas']
dataH15 = go.Scatter(x=dfH15[X], y=dfH15[Y], mode='markers', orientation=ORI,
                    marker_color= 'black', marker_symbol= 'star-square-open',  marker_size =markerSize,
                    name = 'Flue Gas',
                    )

#ECR - iHN637 - Methanogensis

dfH10 = df.loc[df[S] == 'Methanol_MethylTransferase_1']
dataH10 = go.Scatter(x=dfH10[X], y=dfH10[Y], mode='markers', orientation=ORI,
                    marker_color= 'dodgerblue', marker_symbol= 'star-open', marker_size = markerSize+4,
                    name = 'Methanol Fixation 1 ATP',    
                    )

dfH11 = df.loc[df[S] == 'Methanol_MethylTransferase_0.5']
dataH11 = go.Scatter(x=dfH11[X], y=dfH11[Y], mode='markers', orientation=ORI,
                    marker_color= 'red', marker_symbol= 'star-open', marker_size = markerSize+3,
                    name = 'Methanol Fixation 0.5 ATP',    
                    )

dfH12 = df.loc[df[S] == 'Methanol_MethylTransferase_0.1']
dataH12 = go.Scatter(x=dfH12[X], y=dfH12[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkgreen', marker_symbol= 'star-open', marker_size = markerSize+2,
                    name = 'Methanol Fixation 0.1 ATP',    
                    )

dfH13 = df.loc[df[S] == 'Methanol_MethylTransferase_0']
dataH13 = go.Scatter(x=dfH13[X], y=dfH13[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkviolet', marker_symbol= 'star-open', marker_size = markerSize+1,
                    name = 'Methanol Fixation no ATP',    
                    )
                    

data = [dataC21, dataC22, dataC23, dataC24, dataC25, dataC26, dataC27,
        dataC11, dataC12, dataC13, dataC14, dataC15, dataC16, dataC31,
        dataH1, dataH2, dataH3, dataH4, dataH5, dataH6, dataH7, dataH8, dataH9,
        dataS1, dataS2, dataS3, dataH10, dataH11, dataH12, dataH13, dataH14, ]
        #dataH15
        

#shape1=dict(type="line", x0=1, x1=1, y0=-1, y1=57, xref='x', yref='y', line=dict(color='black', width=1.5))
#shape2=dict(type="rect", x0=6, x1=10, y0=0, y1=2.5, xref='x', yref='y',fillcolor='orange', opacity=0.2, line=dict(color='darkorange', width=2))
#print(data)

#shapes = [shape1]

layout = go.Layout(#title_text='Bioproduction Cost ($CA/tonne)', title = {'x': title_pos},
                   width=W, height=H, template="simple_white", plot_bgcolor = "white",
                   margin=go.layout.Margin(l=0, r=20,b=0, t=50),
                  legend_font_size=16,)
                  #shapes=shapes)

fig = go.Figure(data = data, layout = layout)



if ORI == 'v':
        fig.update_xaxes(title_text = 'Bioproduct', title_font_size= 16, tickfont_size=14, tickangle=-90,
                 showline=True, linewidth=1, color='black', mirror = True,
                 showgrid=True, gridcolor='#eee',
                 range=[-1,n])
        fig.update_yaxes(title_text = 'Cost of Production ($CA/tonne)', title_font_size= 16, tickfont_size=16,
                 showline=True, linewidth=1, color='black', mirror = True, showgrid=True, gridcolor='#eee', type='log', #)
                        ) #range=[2.7, 4.5],) #tick0 = 0, dtick= 250)
        
    
    
elif ORI == 'h':
    # Specify major tick positions and labels
    major_tickvals = [1,10, 100, 1000, 10000]
    major_ticktext = ['1','10', '100', '1000', '10000']

    # Specify minor tick positions and labels
    minor_tickvals = []
    minor_ticktext = []
    for i in [1,2,5]:
        for major_tick in major_tickvals:
          minor_tickvals.append(major_tick * i)
          minor_ticktext.append(str(major_tick * i))

    fig.update_xaxes(title_text = 'Cost of Production ($CA/tonne)', title_font_size= 20, tickfont_size=15,
                showline=True, linewidth=1, color='black', mirror = True, showgrid=True, gridcolor='#eee',
                type='log', range=[0.0, 4.0], dtick = 'D2' ,tickvals=major_tickvals + minor_tickvals, ticktext=major_ticktext + minor_ticktext) #2.7 for base case
    fig.update_yaxes(title_text = 'Bioproducts', title_font_size= 20, tickfont_size=15,
                 showline=True, linewidth=1, color='black', mirror = True,
                 showgrid=True, gridcolor='#eee',
                 range=[-1,n]) 
    
    #fig.update_layout(legend=dict(yanchor="top", y=0.995, xanchor="right", x=0.99)) #Comment out to move legend outside graph

fig.show()
fig.write_image(outputDirectory_general + date + "_ECR-BioproductionCost_Scatter-" +  ORI +  "_scenario-" + scenario + "_case-" + case  + "-LOG.png", scale=5)

<>:13: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:21: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:23: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:26: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:34: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:36: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:13: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:21: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:23: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:26: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:34: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:36: SyntaxWarning:

"is" with a literal. Did you mean "=="?

C:\Users\austi\AppData\Local\Temp\ipykernel_23548\2678935581.py:13: SyntaxWarning:

"is" with a literal. Did you mean "=="?

C:\Users\austi\AppData\Local\Temp\ipykernel_23548\2678935581.py:21: SyntaxWarning:

"is" with a literal. Di

## Bio Analysis ##

**Notes:**
- FBA-predicted yields

In [13]:
FileName = 'FBA_OverallSummary-UPDATED_2021-6-10_AZ.xlsx'
SheetName = 'Overall-Results'
Type = 'Bio'

df = readInData(FileName, SheetName, Type)
substrates_to_remove = ['Glucose', 'Xylose', 'Glycerol']
filtered_df = df[~df['Substrate'].isin(substrates_to_remove)]
display(filtered_df)
df = filtered_df

,Date,BaseModel,Substrate,Substrate_Type,Product,Objective,S_ExchangeLB,O2_ExchangeLB,H2_ExchangeLB,ATPM_LB,...,Charge_P,Deg_Red_P,MW_g/mol_P,BP_degC_P,MoleculeType_P,CCM_Feeder_P,Transport_P,MetID_P,PathwayGraph,SubstrateType
0,2021-05-11,iJO1366,Acetaldehyde,ECR-C2,Malonate,EX_malon_e,-10,-20,NaN,3.15,...,0,2.0,102.04378,199,Dicarboxylate,Oxaloacetate,Proton symporter,malon_c,Acetaldehyde,C2
1,2021-05-11,iJO1366,Acetate,ECR-C2,Malonate,EX_malon_e,-10,-20,NaN,3.15,...,0,2.0,102.04378,199,Dicarboxylate,Oxaloacetate,Proton symporter,malon_c,Acetic Acid,C2
2,2021-05-11,iJO1366,EG_Glycolate,ECR-C2,Malonate,EX_malon_e,-10,-20,NaN,3.15,...,0,2.0,102.04378,199,Dicarboxylate,Oxaloacetate,Proton symporter,malon_c,EG - Glycolate,C2
3,2021-05-11,iJO1366,EG_SACA,ECR-C2,Malonate,EX_malon_e,-10,-20,NaN,3.15,...,0,2.0,102.04378,199,Dicarboxylate,Oxaloacetate,Proton symporter,malon_c,EG - SACA,C2
4,2021-05-11,iJO1366,Ethanol,ECR-C2,Malonate,EX_malon_e,-10,-20,NaN,3.15,...,0,2.0,102.04378,199,Dicarboxylate,Oxaloacetate,Proton symporter,malon_c,Ethanol,C2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1569,2023-12-14,iHN637,CO_2:H2_2,ECR-C1b,Limonene,EX_limnen_e,-10,0,-20.0,0.45,...,0,5.6,136.23244,176,Isoprenoid; Aromatic (Cycloalkane),Acetyl-CoA,Proton symporter,limnen_c,1 CO2:X H2,CO2+H2
1570,2023-12-14,iHN637,CO_2:H2_2,ECR-C1b,Malate_L,EX_mal__L_e,-10,0,-20.0,0.45,...,-2,2.5,132.06916,NaN,Dicarboxylate,Oxaloacetate,Proton symporter,mal__L_e,1 CO2:X H2,CO2+H2
1571,2023-12-14,iHN637,CO_2:H2_2,ECR-C1b,Succinate,EX_succ_e,-10,0,-20.0,0.45,...,-2,3.0,116.07016,235,Dicarboxylate,Oxaloacetate,Proton symporter,succ_c,1 CO2:X H2,CO2+H2
1572,2023-12-14,iHN637,CO_2:H2_2,ECR-C1b,Glycolate,EX_glyclt_e,-10,0,-20.0,0.45,...,-1,2.5,75.04192,112,Carboxylate + Alcohol (hydroxy carboxylate),Glyoxylate (*isocitrate),Proton symporter,glyclt_c,1 CO2:X H2,CO2+H2


### Bio - Box Plot ###

In [19]:
##----- Bioproduction Yields Box Plot -----##

ORI = 'h' #'v' for vertical, 'h' for horizontal

if ORI is 'v':
    X = 'PathwayGraph'
    Y = 'Yield_Mass'
    W = 900
    H = 500
    
elif ORI is 'h':
    X = 'Yield_Mass'
    Y = 'PathwayGraph'
    W = 900
    H = 900

    
#layout = dict(xaxis = dict(title = 'Substrate test', showgrid=False, ticks='inside'))
    
fig = px.box(df, x=X, y=Y,
             orientation=ORI,
             color = 'SubstrateType', color_discrete_map={'C2': "Red", 'C1': 'Blue', 'C1+H2': 'Green'},#, 'Sugar': 'Purple'},
             #title = 'Bioproduction Mass Yields',
             points = 'outliers',
             category_orders={'PathwayGraph':['Acetaldehyde', 'Acetic Acid', 'Ethanol', 'EG - Glycolate', 'EG - SACA',
             'GlycAld - Glycolate', 'GlycAld - SACA', 'Formate - Formolase', 'Formate - RGP', 'Formate - Serine Cycle', 
             'Methanol - Formolase', 'Methanol - RuMP', 'Methanol - Serine Cycle', 
             '1 CO:0 H2', '1 CO:0.5 H2', '1 CO:1 H2', '1 CO:2 H2', 
             '1 FOR:0 H2', '1 FOR:0.5 H2', '1 FOR:1 H2', '1 FOR:2 H2','1 CO2:X H2',
             'Methanol Fixation - 1 ATP','Methanol Fixation - 0.5 ATP','Methanol Fixation - 0.1 ATP','Methanol Fixation - no ATP'],
                 #,'Glucose', 'Glycerol', 'Xylose'],
             'SubstrateType': ['C2', 'C1', 'C1+H2']}
             #range_x =[0,14]
            )


fig.update_layout(template="simple_white", plot_bgcolor = "white", title = {'x': 0.52},
                 width=W, height=H, title_font_size=18, 
                 legend_title_text='Substrate Type', legend_font_size=16, legend_title_font_size=16
                 )

#The following is needed to change the width of the boxes, and ensure they're aligned with the labels
fig.update_traces(width=0.25)


#Horizontal vs Vertical box plot need opposite axis settings:
if ORI is 'v':
    fig.update_xaxes(title_text='Substrate', title_font_size=16,
                     visible=True, showline=True, linewidth=1, color='black', mirror = True,
                     tickfont_size=14,
                     type='category'#, tickson='boundaries'
                     )

    fig.update_yaxes(title_text='Mass Yield (g/g)', title_font_size=16,
                    visible=True, showline=True, linewidth=1, color='black', mirror = True,
                    tickmode = 'linear', tick0 = 0, dtick= 0.2, tickfont_size=14,
                    showgrid=True, gridcolor='#eee'
                    )
    
    
elif ORI is 'h':
    
    fig.update_yaxes(title_text='Substrate', title_font_size=16,
                 visible=True, showline=True, linewidth=1, color='black', mirror = True,
                 tickfont_size=16,
                 type='category',#, tickson='boundaries'
                 range=[-1,26]    
                 )
                 #tickson="boundaries")

    fig.update_xaxes(title_text='Mass Yield (g/g)', title_font_size=16,
                visible=True, showline=True, linewidth=1, color='black', mirror = True,
                tickmode = 'linear', tick0 = 0, dtick= 0.2, tickfont_size=16,
                showgrid=True, gridcolor='#eee'
                )
    
    shape1=dict(type="line", x0=1, x1=1, y0=-1, y1=28, xref='x', yref='y', line=dict(color='black', width=1.5))
    shapes = [shape1]
    fig.update_layout(legend=dict(yanchor="bottom", y=0.01, xanchor="right", x=0.99), shapes=shapes)

#fig.update_traces(texttemplate='%{text:.2s}', textposition='outside')

    
    #category_orders={'Substrate':['C2', 'C1', 'C1+H2']}
#color_discrete_sequence=px.colors.qualitative.Set1,
#"C1":['CarbonMonoxide','FormicAcid','Methanol']
#for t in fig.data:
   # t.marker.line.width = 1
    #t.marker.line.color = 'black'


fig.show()
fig.write_image(outputDirectory_general + date + "_BioproductionYields-BOX_" + ORI + ".png", scale=5)

<>:5: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:11: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:47: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:61: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:5: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:11: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:47: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:61: SyntaxWarning:

"is" with a literal. Did you mean "=="?

C:\Users\austi\AppData\Local\Temp\ipykernel_24412\799421822.py:5: SyntaxWarning:

"is" with a literal. Did you mean "=="?

C:\Users\austi\AppData\Local\Temp\ipykernel_24412\799421822.py:11: SyntaxWarning:

"is" with a literal. Did you mean "=="?

C:\Users\austi\AppData\Local\Temp\ipykernel_24412\799421822.py:47: SyntaxWarning:

"is" with a literal. Did you mean "=="?

C:\Users\austi\AppData\Local\Temp\ipykernel_24412\799421822.py:61: SyntaxWarning:

"is" with a literal. Did you mean "=

Bio - Discussion Plot B

In [5]:
FileName = 'FBA_OverallSummary-ForDiscussionB_AZ.xlsx'
SheetName = 'Overall-Results'
Type = 'Bio'

df = readInData(FileName, SheetName, Type)
display(df)

,Unnamed: 0,Substrate,Formula_S,C_Sn,H_Sn,O_Sn,MW_S,N_electrons,ParametersType,CellVoltage,...,Flue_gas_SP,Bioreactor_SP,Carbon_capture_rec,CC_1rec,CC_2rec,CC_3rec,CC_4rec,Flue_gas_rec,Bioreactor_rec,SubstrateType
0,105,Methanol,CH3OH,1,4,1,32.04106,6,Categories-Category_2,1.213,...,504.131987,444.261161,659.796136,5987.082647,3.209736,0.0,0.000000,504.131987,444.261161,C1
1,74,Methanol,CH3OH,1,4,1,32.04106,6,Categories-Category_2,1.213,...,491.416451,433.055725,643.154341,5836.072661,3.128778,0.0,0.000000,491.416451,433.055725,C1
2,73,Methanol,CH3OH,1,4,1,32.04106,6,Categories-Category_2,1.213,...,491.200301,432.865244,642.871448,5833.505654,3.127402,0.0,0.000000,491.200301,432.865244,C1
3,104,Methanol,CH3OH,1,4,1,32.04106,6,Categories-Category_2,1.213,...,460.351067,405.679672,602.496693,5467.139467,2.930989,0.0,0.000000,460.351067,405.679672,C1
4,72,Methanol,CH3OH,1,4,1,32.04106,6,Categories-Category_2,1.213,...,437.238944,385.312350,572.248087,5192.659377,2.783838,0.0,0.000000,437.238944,385.312350,C1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1506,1506,FormicAcid,CHOOH,1,1,2,45.01654,2,Categories-Category_2,1.400,...,544.716817,386.435739,575.803298,1195.633886,2.448445,0.0,5121.365887,544.716817,532.760478,C1
1507,1507,FormicAcid,CHOOH,1,1,2,45.01654,2,Categories-Category_2,1.400,...,767.661775,605.639898,588.715295,1197.103705,2.506311,0.0,5251.779410,557.590599,545.619562,C1
1508,1508,FormicAcid,CHOOH,1,1,2,45.01654,2,Categories-Category_2,1.400,...,776.542606,612.691614,594.963197,1195.299486,2.534606,0.0,5316.429902,563.885410,551.932416,C1
1509,1509,FormicAcid,CHOOH,1,1,2,45.01654,2,Categories-Category_2,1.400,...,805.915639,635.986560,615.980442,1199.137849,2.628627,0.0,5527.819516,584.802858,572.811479,C1


In [6]:
##----- Bioproduction Energy Input Box Plot -----##
from sklearn.preprocessing import MinMaxScaler

# Filter rows with the substrate
filtered_df = df[df["SubstratePathway"] == "Glycolaldehyde_SACA"]

min_max_scaler = MinMaxScaler()

filtered_df = filtered_df[filtered_df['YieldBP_Mass'] != 0]
filtered_df = filtered_df[filtered_df['Carbon_Efficiency'] != 0]
filtered_df = filtered_df[filtered_df['Energy_Input'] != 0]

#filtered_df.loc[:,"Carbon_Efficiency"] = min_max_scaler.fit_transform(filtered_df[["Carbon_Efficiency"]])
#filtered_df.loc[:,"YieldBP_Mass"] = min_max_scaler.fit_transform(filtered_df[["YieldBP_Mass"]])
filtered_df.loc[:,"Energy_Input"] = min_max_scaler.fit_transform(filtered_df[["Energy_Input"]])

# Create a box plot using Plotly Express
fig = px.box(filtered_df, 
             y=[ "YieldBP_Mass","Carbon_Efficiency", "Energy_Input"],
             #title="Anaerobic Methanol Fixation",
             color_discrete_sequence=["blue"]
             #points="all"  # Show all data points
            )

fig.update_layout(template="simple_white", plot_bgcolor = "white", title = {'x': 0.52},
                 width=300, height=600, title_font_size=18, 
                 legend_title_text='Substrate Type', legend_font_size=16, legend_title_font_size=16
                 )

fig.update_layout(xaxis=dict(title="", tickvals=[0, 1, 2], ticktext=["Yield","CE", "EI"]),yaxis=dict(title=''))
fig.update_yaxes(range=[0, 1.7])

# Show the plot
fig.show()


fig.write_image(outputDirectory_general + date + "_DiscussionB_Glycolaldehyde_SACA" + ".png", scale=5)

In [60]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Filter rows with the substrate
filtered_df = df[df["SubstratePathway"] == "Glycolaldehyde_SACA"]


filtered_df = filtered_df[filtered_df['YieldBP_Mass'] != 0]
filtered_df = filtered_df[filtered_df['Carbon_Efficiency'] != 0]
filtered_df = filtered_df[filtered_df['Energy_Input'] != 0]

# Create subplot
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Add the first subplot with two box plots sharing the same y-axis
fig.add_trace(go.Box(y=filtered_df["YieldBP_Mass"], name="Yield"), secondary_y=False)
fig.add_trace(go.Box(y=filtered_df["Carbon_Efficiency"], name="Carbon Efficiency"), secondary_y=False)

# Add the second subplot with the third box plot and its own y-axis
fig.add_trace(go.Box(y=filtered_df["Energy_Input"], name="Energy Input", yaxis="y2"), secondary_y=True)

fig.update_traces(showlegend=False)

# Update layout
fig.update_layout(xaxis=dict(title="", tickvals=[0, 1, 2], ticktext=["Yield","CE", "EI"]),yaxis=dict(title=''))
fig.update_yaxes(secondary_y=False,range=[0, 1.7])
fig.update_yaxes(title_text="GJ/tonne", secondary_y=True,range=[0, 100])

fig.update_layout(template="simple_white", plot_bgcolor = "white", title = {'x': 0.52},
                 width=250, height=600, title_font_size=18, 
                 )
# Show the plot
fig.show()


fig.write_image(outputDirectory_general + date + "_DiscussionBv3_Glycolaldehyde_SACA" + ".png", scale=5)

In [47]:
##----- Discussion Figure B Alternative -----##

#df = df.loc[df['MoleculeType_P'] == 'Hydrocarbon (alkane)']
#df = df.sort_values(by='C_P', ascending=0)
df = df.sort_values(by='Deg_RedBP', ascending=0)
#display(df)

df = df[df['YieldBP_Mass'] != 0]
df = df[df['Carbon_Efficiency'] != 0]
df = df[df['Energy_Input'] != 0]

S = 'SubstratePathway'

ORI = 'h'
X = 'YieldBP_Mass'
Y = 'BioProduct_Graph'
W = 700
H = 1100
title_pos = 0.6

markerSize=7

fig = go.Figure()

#ECR 1-C compounds; iJO1366
df1 = df.loc[df[S] == 'Methanol_Formolase']
data1 = go.Scatter(x=df1[X], y=df1[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkviolet', marker_symbol= 'square', marker_size = markerSize+1,
                    name = 'MeOH - Formolase',    
                    )

df2 = df.loc[df[S] == 'Methanol_RuMP']
data2 = go.Scatter(x=df2[X], y=df2[Y], mode='markers', orientation=ORI,
                    marker_color= 'deepskyblue', marker_symbol= 'square', marker_size = markerSize-1,
                    name = 'MeOH - RuMP',
                    )

df3 = df.loc[df[S] == 'Methanol_SerineCycle']
data3 = go.Scatter(x=df3[X], y=df3[Y], mode='markers', orientation=ORI,
                    marker_color= 'red', marker_symbol= 'square', marker_size = markerSize,
                    name = 'MeOH - SerineCycle'    
                    )
                   

df4 = df.loc[df[S] == 'EG_Glycolate']
data4 = go.Scatter(x=df4[X], y=df4[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkblue', marker_symbol= 'triangle-up', marker_size =markerSize,
                    name = 'EG - Glycolate',
                    )

df5 = df.loc[df[S] == 'EG_SACA']
data5 = go.Scatter(x=df5[X], y=df5[Y],mode='markers', orientation=ORI,
                    marker_color= 'red', marker_symbol= 'triangle-up', marker_size =markerSize,
                    name = 'EG - SACA',
                    )



df6 = df.loc[df[S] == 'Glycolaldehyde_Glycolate']
data6 = go.Scatter(x=df6[X], y=df6[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkorange', marker_symbol= 'triangle-up', marker_size =markerSize,
                    name = 'GlyAld - Glycolate',
                    )

df7 = df.loc[df[S] == 'Glycolaldehyde_SACA']
data7 = go.Scatter(x=df7[X], y=df7[Y], mode='markers', orientation=ORI,
                    marker_color= 'deepskyblue', marker_symbol= 'triangle-up', marker_size =markerSize,
                    name = 'GlyAld - SACA',
                    )
                    
df8 = df.loc[df[S] == 'Methanol_MethylTransferase_0']
data8 = go.Scatter(x=df8[X], y=df8[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkviolet', marker_symbol= 'star-open', marker_size = markerSize+1,
                    name = 'Methanol Fixation no ATP',    
                    )
                    

data = [data1,data2,data3,data4,data5,data6,data7,data8]
        

if ORI is 'v':
    shape1=dict(type="line", x0=-1, x1=57, y0=1, y1=1, xref='x', yref='y', line=dict(color='black', width=1.5))
elif ORI is 'h':
    shape1=dict(type="line", x0=1, x1=1, y0=-1, y1=57, xref='x', yref='y', line=dict(color='black', width=1.5))
#shape2=dict(type="rect", x0=6, x1=10, y0=0, y1=2.5, xref='x', yref='y',fillcolor='orange', opacity=0.2, line=dict(color='darkorange', width=2))


shapes = [shape1]

layout = go.Layout(title_text='Bioproduction Yield', title = {'x': title_pos},
                   width=W, height=H, template="simple_white", plot_bgcolor = "white",
                   margin=go.layout.Margin(l=0, r=20,b=0, t=50),
                  legend_font_size=14.5 #20 #for ORI=h,#)
                  ,shapes=shapes)

fig = go.Figure(data = data, layout = layout)
  
    
fig.update_xaxes(title_text = 'Bioproduction Yield (tonne/tonne)', title_font_size= 12, tickfont_size=12,
                showline=True, linewidth=1, color='black', mirror = True, showgrid=True, gridcolor='#eee',
            range=[0,1.7]) #, tick0 = 0, dtick= 0.2)
fig.update_yaxes(title_text = 'Bioproducts', title_font_size= 12, tickfont_size=12,
                showline=True, linewidth=1, color='black', mirror = True,
                showgrid=True, gridcolor='#eee',
                range=[-1,57]) 

fig.update_layout(legend=dict(yanchor="bottom", y=0.01, xanchor="right", x=0.99),showlegend=False)

fig.show()
fig.write_image(outputDirectory_general + date + "_figureB_Alt_Y" + ".png", scale=5)
fig.write_html(outputDirectory_general + date + "_figureB_Alt_Y" + ".html")

<>:81: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:83: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:81: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:83: SyntaxWarning:

"is" with a literal. Did you mean "=="?

C:\Users\austi\AppData\Local\Temp\ipykernel_9604\1810358336.py:81: SyntaxWarning:

"is" with a literal. Did you mean "=="?

C:\Users\austi\AppData\Local\Temp\ipykernel_9604\1810358336.py:83: SyntaxWarning:

"is" with a literal. Did you mean "=="?



### Bio - Carbon Efficiency ###

In [8]:
##----- Bioproduction Carbon Efficiency -----##

#df = df.loc[df['MoleculeType_P'] == 'Hydrocarbon (alkane)']
#df = df.sort_values(by='C_P', ascending=0)
df = df.sort_values(by='Deg_RedBP', ascending=0)
#df = df[df['Deg_RedBP'] <= 5.601] # For sorting chems/fuels
#display(df)


ORI = 'h' #'v' for vertical, 'h' for horizontal
S = 'SubstratePathway'

if ORI is 'v':
    X = 'BioProduct_Graph'
    Y = 'Carbon_Efficiency'
    W = 1400
    H = 800
    title_pos = 0.45
    #For glucose costs:
    X2 = X
    if case is 'Category_2':
        Y2 = 'Cost_Best'
    elif case is 'Category_1':
        Y2 = 'Cost_Base'    
elif ORI is 'h':
    X = 'Carbon_Efficiency'
    Y = 'BioProduct_Graph'
    W = 900
    H = 900
    title_pos = 0.6
    #For glucose costs:
    Y2 = Y
    if case is 'Category_2':
        X2 = 'Cost_Best'
    elif case is 'Category_1':
        X2 = 'Cost_Base'

markerSize=7

fig = go.Figure()

dfS1 = df2.loc[df2[S] == 'Glucose']
dataS1 = go.Scatter(x=dfS1[X2], y=dfS1[Y2], mode='markers', orientation=ORI,
                    marker_color= 'gold', marker_line_color='black', marker_line_width=0.75, marker_symbol= 'diamond', marker_size=markerSize,
                    name = 'Glucose'
                    )

dfS2 = df.loc[df[S] == 'Glycerol']
dataS2 = go.Scatter(x=dfS2[X], y=dfS2[Y], mode='markers', orientation=ORI,
                    marker_color= 'red',marker_symbol= 'diamond', marker_size=markerSize,
                    name = 'Glycerol'
                    )

dfS3 = df.loc[df[S] == 'Xylose'] 
dataS3 = go.Scatter(x=dfS3[X], y=dfS3[Y], mode='markers', orientation=ORI,
                    marker_color= 'blue', marker_symbol= 'diamond', marker_size=markerSize,
                    name = 'Xylose'
                    )

#ECR 1-C compounds; iJO1366
dfC11 = df.loc[df[S] == 'Methanol_Formolase']
dataC11 = go.Scatter(x=dfC11[X], y=dfC11[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkviolet', marker_symbol= 'square', marker_size = markerSize+1,
                    name = 'MeOH - Formolase',    
                    )

dfC12 = df.loc[df[S] == 'Methanol_RuMP']
dataC12 = go.Scatter(x=dfC12[X], y=dfC12[Y], mode='markers', orientation=ORI,
                    marker_color= 'deepskyblue', marker_symbol= 'square', marker_size = markerSize-1,
                    name = 'MeOH - RuMP',
                    )

dfC13 = df.loc[df[S] == 'Methanol_SerineCycle']
dataC13 = go.Scatter(x=dfC13[X], y=dfC13[Y], mode='markers', orientation=ORI,
                    marker_color= 'red', marker_symbol= 'square', marker_size = markerSize,
                    name = 'MeOH - SerineCycle'    
                    )

dfC14 = df.loc[df[S] == 'Formate_Formolase']
dataC14 = go.Scatter(x=dfC14[X], y=dfC14[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkviolet', marker_symbol= 'x', marker_size = markerSize,
                    name = 'Formate - Formolase',
                    )

dfC15 = df.loc[df[S] == 'Formate_RGP-Serine']
dataC15 = go.Scatter(x=dfC15[X], y=dfC15[Y],mode='markers', orientation=ORI,
                    marker_color= 'olivedrab', marker_symbol= 'x', marker_size = markerSize,
                    name = 'Formate - RGP',
                    )

dfC16 = df.loc[df[S] == 'Formate_SerineCycle']
dataC16 = go.Scatter(x=dfC16[X], y=dfC16[Y],mode='markers', orientation=ORI,
                    marker_color= 'red', marker_symbol= 'x', marker_size = markerSize,
                    name = 'Formate - Serine Cycle',
                    )

#ECR 2-C compounds; iJO1366
dfC21 = df.loc[df[S] == 'Acetaldehyde']
dataC21 = go.Scatter(x=dfC21[X], y=dfC21[Y], mode='markers', orientation=ORI,
                    marker_color='darkblue', marker_symbol= 'circle', marker_size =markerSize,
                    name = 'Acetaldehyde',
                    )

dfC22 = df.loc[df[S] == 'Acetate']
dataC22 = go.Scatter(x=dfC22[X], y=dfC22[Y], mode='markers', orientation=ORI,
                    marker_color= 'red', marker_symbol= 'circle', marker_size =markerSize,
                    name = 'Acetate',
                    )


dfC23 = df.loc[df[S] == 'Ethanol']
dataC23 = go.Scatter(x=dfC23[X], y=dfC23[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkorange',marker_symbol= 'circle', marker_size =markerSize,
                    name = 'Ethanol',
                    )

dfC24 = df.loc[df[S] == 'EG_Glycolate']
dataC24 = go.Scatter(x=dfC24[X], y=dfC24[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkblue', marker_symbol= 'triangle-up', marker_size =markerSize,
                    name = 'EG - Glycolate',
                    )

dfC25 = df.loc[df[S] == 'EG_SACA']
dataC25 = go.Scatter(x=dfC25[X], y=dfC25[Y],mode='markers', orientation=ORI,
                    marker_color= 'red', marker_symbol= 'triangle-up', marker_size =markerSize,
                    name = 'EG - SACA',
                    )



dfC26 = df.loc[df[S] == 'Glycolaldehyde_Glycolate']
dataC26 = go.Scatter(x=dfC26[X], y=dfC26[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkorange', marker_symbol= 'triangle-up', marker_size =markerSize,
                    name = 'GlyAld - Glycolate',
                    )

dfC27 = df.loc[df[S] == 'Glycolaldehyde_SACA']
dataC27 = go.Scatter(x=dfC27[X], y=dfC27[Y], mode='markers', orientation=ORI,
                    marker_color= 'deepskyblue', marker_symbol= 'triangle-up', marker_size =markerSize,
                    name = 'GlyAld - SACA',
                    )
#ECR 3-C compounds; iJO1366
dfC31 = df.loc[df[S] == 'N_Propanol']
dataC31 = go.Scatter(x=(dfC31[X]*2/3), y=dfC31[Y], mode='markers', orientation=ORI,
                    marker_color= 'black', marker_symbol= 'circle', marker_size =markerSize,
                    name = 'n-Propanol',
                    ) # The *(2/3) modification is necessary for the carbon efficiency as for now I've assumed propanol as "ethanol" pricing, thus in the calculations it assumes 2 mols C instead of 3

#ECR - iHN637 - CO + Formate Mixes
dfH1 = df.loc[df[S] == 'FOR_1:H2_0'] 
dataH1 = go.Scatter(x=dfH1[X], y=dfH1[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkgreen', marker_symbol= 'star-square-open', marker_size =markerSize,
                    name = '1 FOR:0 H2',
                    )

dfH2 = df.loc[df[S] == 'FOR_1:H2_0.5']
dataH2 = go.Scatter(x=dfH2[X], y=dfH2[Y], mode='markers', orientation=ORI,
                    marker_color= 'red', marker_symbol= 'star-square-open', marker_size =markerSize,
                    name = '1 FOR:0.5 H2',
                    )

dfH3 = df.loc[df[S] == 'FOR_1:H2_1']
dataH3 = go.Scatter(x=dfH3[X], y=dfH3[Y], mode='markers', orientation=ORI,
                    marker_color= 'dodgerblue', marker_symbol= 'star-square-open', marker_size =markerSize,
                    name = '1 FOR:1 H2',
                    )

dfH4 = df.loc[df[S] == 'FOR_1:H2_2']
dataH4 = go.Scatter(x=dfH4[X], y=dfH4[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkviolet', marker_symbol= 'star-square-open', marker_size =markerSize,
                    name = '1 FOR:2 H2',
                    )

dfH5 = df.loc[df[S] == 'CO_1:H2_0']
dataH5 = go.Scatter(x=dfH5[X], y=dfH5[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkgreen', marker_symbol= 'diamond-tall-open', marker_size =markerSize+0.5,
                    name = '1 CO:0 H2',
                    )

dfH6 = df.loc[df[S] == 'CO_1:H2_0.5'] 
dataH6 = go.Scatter(x=dfH6[X], y=dfH6[Y], mode='markers', orientation=ORI,
                    marker_color= 'red', marker_symbol= 'diamond-tall-open', marker_size =markerSize+0.5,
                    name = '1 CO:0.5 H2',
                    )

dfH7 = df.loc[df[S] == 'CO_1:H2_1']
dataH7 = go.Scatter(x=dfH7[X], y=dfH7[Y], mode='markers', orientation=ORI,
                    marker_color= 'dodgerblue', marker_symbol= 'diamond-tall-open', marker_size =markerSize+0.5,
                    name = '1 CO:1 H2',
                    )

dfH8 = df.loc[df[S] == 'CO_1:H2_2']
dataH8 = go.Scatter(x=dfH8[X], y=dfH8[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkviolet', marker_symbol= 'diamond-tall-open',  marker_size =markerSize+0.5,
                    name = '1 CO:2 H2',
                    )

dfH9 = df.loc[df[S] == 'CO_2:H2_3']
dataH9 = go.Scatter(x=dfH9[X], y=dfH9[Y], mode='markers', orientation=ORI,
                    marker_color= 'black', marker_symbol= 'circle-open',  marker_size =markerSize,
                    name = '1 CO2:X H2',
                    )

dfH14 = df.loc[df[S] == 'CO_2:CO1_4']
dataH14 = go.Scatter(x=dfH14[X], y=dfH14[Y], mode='markers', orientation=ORI,
                    marker_color= 'black', marker_symbol= 'diamond-tall-open',  marker_size =markerSize,
                    name = '1 CO2:4 CO',
                    )

dfH15 = df.loc[df[S] == 'Flue_Gas']
dataH15 = go.Scatter(x=dfH15[X], y=dfH15[Y], mode='markers', orientation=ORI,
                    marker_color= 'black', marker_symbol= 'star-square-open',  marker_size =markerSize,
                    name = 'Flue Gas',
                    )


#ECR - iHN637 - Methanogensis

dfH10 = df.loc[df[S] == 'Methanol_MethylTransferase_1']
dataH10 = go.Scatter(x=dfH10[X], y=dfH10[Y], mode='markers', orientation=ORI,
                    marker_color= 'dodgerblue', marker_symbol= 'star-open', marker_size = markerSize+4,
                    name = 'Methanol Fixation 1 ATP',    
                    )

dfH11 = df.loc[df[S] == 'Methanol_MethylTransferase_0.5']
dataH11 = go.Scatter(x=dfH11[X], y=dfH11[Y], mode='markers', orientation=ORI,
                    marker_color= 'red', marker_symbol= 'star-open', marker_size = markerSize+3,
                    name = 'Methanol Fixation 0.5 ATP',    
                    )

dfH12 = df.loc[df[S] == 'Methanol_MethylTransferase_0.1']
dataH12 = go.Scatter(x=dfH12[X], y=dfH12[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkgreen', marker_symbol= 'star-open', marker_size = markerSize+2,
                    name = 'Methanol Fixation 0.1 ATP',    
                    )

dfH13 = df.loc[df[S] == 'Methanol_MethylTransferase_0']
dataH13 = go.Scatter(x=dfH13[X], y=dfH13[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkviolet', marker_symbol= 'star-open', marker_size = markerSize+1,
                    name = 'Methanol Fixation no ATP',    
                    )
                    

data = [dataC21, dataC22, dataC23, dataC24, dataC25, dataC26, dataC27,
        dataC11, dataC12, dataC13, dataC14, dataC15, dataC16, dataC31,
        dataH1, dataH2, dataH3, dataH4, dataH5, dataH6, dataH7, dataH8, dataH9,
        dataH10, dataH11, dataH12, dataH13, dataH14] #, dataH15]

#for trace in data:
#    trace.visible = 'legendonly'

if ORI is 'v':
    shape1=dict(type="line", x0=-1, x1=57, y0=1, y1=1, xref='x', yref='y', line=dict(color='black', width=1.5))
elif ORI is 'h':
    shape1=dict(type="line", x0=1, x1=1, y0=-1, y1=57, xref='x', yref='y', line=dict(color='black', width=1.5))
#shape2=dict(type="rect", x0=6, x1=10, y0=0, y1=2.5, xref='x', yref='y',fillcolor='orange', opacity=0.2, line=dict(color='darkorange', width=2))


shapes = [shape1]

layout = go.Layout(title_text='Bioproduction Carbon Efficiency (Chemicals)', title = {'x': title_pos},
                   width=W, height=H, template="simple_white", plot_bgcolor = "white",
                   margin=go.layout.Margin(l=0, r=20,b=0, t=50),
                  legend_font_size=14.5, #20 #for ORI=h,#)
                  shapes=shapes)

fig = go.Figure(data = data, layout = layout)



if ORI == 'v':
        fig.update_xaxes(title_text = 'Bioproducts', title_font_size= 16, tickfont_size=14, tickangle=-90,
                 showline=True, linewidth=1, color='black', mirror = True,
                 showgrid=True, gridcolor='#eee',
                 range=[-1,57])
        fig.update_yaxes(title_text = 'Carbon Efficiency (mol/mol)', title_font_size= 16, tickfont_size=14,
                 showline=True, linewidth=1, color='black', mirror = True, showgrid=True, gridcolor='#eee',
                 range=[0,2.5])
        
    
    
elif ORI == 'h':

    fig.update_xaxes(title_text = 'Carbon Efficiency (mol/mol)', title_font_size= 20, tickfont_size=13,
                 showline=True, linewidth=1, color='black', mirror = True, showgrid=True, gridcolor='#eee',
                range=[0,2.0], tick0 = 0, dtick= 0.2)
    fig.update_yaxes(title_text = 'Bioproducts', title_font_size= 20, tickfont_size=13,
                 showline=True, linewidth=1, color='black', mirror = True,
                 showgrid=True, gridcolor='#eee',
                 range=[-1,57]) 

    fig.update_layout(legend=dict(yanchor="bottom", y=0.01, xanchor="right", x=0.99), showlegend = True)

fig.show()
fig.write_image(outputDirectory_general + date + "_BioproductionCarbon_Efficiency_Legend" + ORI + ".png", scale=5)
fig.write_html(outputDirectory_general + date + "_BioproductionCarbon_Efficiency" + ORI + ".html")

<>:13: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:21: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:23: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:25: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:33: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:35: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:252: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:254: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:13: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:21: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:23: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:25: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:33: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:35: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:252: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:254: SyntaxWarning:

"is" with a l

### DoR Scatterplot ###

In [25]:
FileName1 = 'FBA_OverallSummary-UPDATED_2024-02-13AZ-MT_NP_CO.xlsx' #FBA data for everything but CO2/H2
FileName2 = 'FBA_OverallSummary_CO2-H2_2022-02-13AZ.xlsx' #FBA data for CO2/H2 mixtures
#From thesis analysis: 'FBA_OverallSummary-UPDATED_2021-6-10.xlsx'
SheetName = 'Overall Results'
Type = 'Bio'

#df1 - contains FBA data for everything but CO2/H2
df1 = readInData(FileName1, SheetName, Type)

#df2 - contains FBA data for CO2/H2 mixtures
df2 = readInData(FileName2, SheetName, Type)
df2 = df2.drop(columns=['y_Cs', 'y_H2', 'S_mwMix', 'S_mw']) #CO2-H2 FBA file contains some extra columns I used to manually validate, remove these before combining the dataframes
df2.drop(df2.loc[df2['Substrate']!='CO2_1:H2_4'].index, inplace=True) #Only using the CO2:H2 mixture ration of 1:4, drop all the other ratios
#display(df2)

#Combine df1 and df2 for df with all data
df = pd.concat([df1,df2])


display(df)

,Date,BaseModel,Substrate,Substrate_Type,Product,Objective,S_ExchangeLB,O2_ExchangeLB,H2_ExchangeLB,ATPM_LB,...,Carbons,LHVMolar,LHVMass,LHVProdMass,DoR_S,Cn_S,CE,PathwayGraph,SubstrateType,Unnamed: 0
0,2021-05-11,iJO1366,Ethanol,ECR-C2,Malonate,EX_malon_e,-10,-20,NaN,3.15,...,3,1230.000648,26.700000,0.0,6.0,2.0,1.583731,Ethanol,C2,NaN
1,2024-02-14,iJO1366,N_Propanol,NaN,Malate_L,EX_mal__L_e,-10,-20,0.0,3.15,...,4,1843.720736,30.680000,0.0,6.0,3.0,1.365114,NaN,NaN,NaN
2,2024-02-14,iJO1366,N_Propanol,NaN,Malonate,EX_malon_e,-10,-20,0.0,3.15,...,3,1843.720736,30.680000,0.0,6.0,3.0,1.311228,NaN,NaN,NaN
3,2021-05-17,iJO1366,Ethanol,ECR-C2,Malate_L,EX_mal__L_e,-10,-20,NaN,3.15,...,4,1230.000648,26.700000,0.0,6.0,2.0,1.540000,Ethanol,C2,NaN
4,2024-02-14,iJO1366,N_Propanol,NaN,Glutamate_L,EX_glu__L_e,-10,-20,0.0,3.15,...,5,1843.720736,30.680000,0.0,6.0,3.0,1.481481,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
225,2022-01-03,iHN637,CO2_1:H2_4,CO2-H2,EthylAcetate,EX_aces_e,-10,0,-40.0,0.45,...,4,NaN,23.925551,NaN,0.0,0.2,6.250000,NaN,NaN,0.0
234,2022-01-03,iHN637,CO2_1:H2_4,CO2-H2,1-3-Butanediol-PW1,EX_r13bdo_e,-10,0,-40.0,0.45,...,4,NaN,23.925551,NaN,0.0,0.2,12.500000,NaN,NaN,1.0
235,2022-01-03,iHN637,CO2_1:H2_4,CO2-H2,2-3-Butanediol,EX_btd_RR_e,-10,0,-40.0,0.45,...,4,NaN,23.925551,NaN,0.0,0.2,7.500000,NaN,NaN,2.0
239,2022-01-03,iHN637,CO2_1:H2_4,CO2-H2,1-3-Butanediol-PW2,EX_r13bdo_e,-10,0,-40.0,0.45,...,4,NaN,23.925551,NaN,0.0,0.2,10.000000,NaN,NaN,2.0


In [27]:
#Degree of Reduction Ratio Calculation


##(1) Substrate Degree of Reduction Corrections for C+H2 mixtures (CO, Formate or CO2 with H2)
#(1a) H2 Ratio for mixtures
df['H2Ratio'] = df['H2_Flux']/df['S_Flux'] #Mol H2/Mol C 
#(1b) Degree of reduction of C+H2 mixtures (DoR of pure carbon source + reducing power provided by H2)
df['DoR_SH2'] = df['DoR_S']+df['H2Ratio']*2/1 #DoR = (4*Cn-2*On+1*Hn-3*Nn+5*Pn)/Cn

#(2) Degree of reduction ratios (DoR Substrate / DoR Product
df['Deg_Red_RatioSP1'] = df['DoR_S']/df['Deg_Red_P'] #(2a) - Pure feedstocks
df['Deg_Red_RatioSP2'] = df['DoR_SH2']/df['Deg_Red_P'] #(2b) - C+H2 mixtures

display(df)
df.to_excel(outputDirectory_general + date + "_dftest" + ORI + ".xlsx")

,Date,BaseModel,Substrate,Substrate_Type,Product,Objective,S_ExchangeLB,O2_ExchangeLB,H2_ExchangeLB,ATPM_LB,...,DoR_S,Cn_S,CE,PathwayGraph,SubstrateType,Unnamed: 0,H2Ratio,DoR_SH2,Deg_Red_RatioSP1,Deg_Red_RatioSP2
0,2021-05-11,iJO1366,Ethanol,ECR-C2,Malonate,EX_malon_e,-10,-20,NaN,3.15,...,6.0,2.0,1.583731,Ethanol,C2,NaN,NaN,NaN,3.000000,NaN
1,2024-02-14,iJO1366,N_Propanol,NaN,Malate_L,EX_mal__L_e,-10,-20,0.0,3.15,...,6.0,3.0,1.365114,NaN,NaN,NaN,-0.000000,6.000000,2.400000,2.400000
2,2024-02-14,iJO1366,N_Propanol,NaN,Malonate,EX_malon_e,-10,-20,0.0,3.15,...,6.0,3.0,1.311228,NaN,NaN,NaN,-0.000000,6.000000,3.000000,3.000000
3,2021-05-17,iJO1366,Ethanol,ECR-C2,Malate_L,EX_mal__L_e,-10,-20,NaN,3.15,...,6.0,2.0,1.540000,Ethanol,C2,NaN,NaN,NaN,2.400000,NaN
4,2024-02-14,iJO1366,N_Propanol,NaN,Glutamate_L,EX_glu__L_e,-10,-20,0.0,3.15,...,6.0,3.0,1.481481,NaN,NaN,NaN,-0.000000,6.000000,1.764706,1.764706
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
225,2022-01-03,iHN637,CO2_1:H2_4,CO2-H2,EthylAcetate,EX_aces_e,-10,0,-40.0,0.45,...,0.0,0.2,6.250000,NaN,NaN,0.0,2.500000,5.000000,0.000000,1.470588
234,2022-01-03,iHN637,CO2_1:H2_4,CO2-H2,1-3-Butanediol-PW1,EX_r13bdo_e,-10,0,-40.0,0.45,...,0.0,0.2,12.500000,NaN,NaN,1.0,2.750000,5.500000,0.000000,0.982143
235,2022-01-03,iHN637,CO2_1:H2_4,CO2-H2,2-3-Butanediol,EX_btd_RR_e,-10,0,-40.0,0.45,...,0.0,0.2,7.500000,NaN,NaN,2.0,2.750000,5.500000,0.000000,1.137931
239,2022-01-03,iHN637,CO2_1:H2_4,CO2-H2,1-3-Butanediol-PW2,EX_r13bdo_e,-10,0,-40.0,0.45,...,0.0,0.2,10.000000,NaN,NaN,2.0,2.750000,5.500000,0.000000,0.880000


In [30]:
ORI = 'v' #'v' for vertical, 'h' for horizontal

if ORI is 'v':
    X1 = 'Deg_Red_RatioSP1'
    X2 = 'Deg_Red_RatioSP2'
    Y = 'Yield_Mass'
    W = 800
    H = 800
    title_pos = 0.45
    
    
markerSize=7

fig = go.Figure()
#Pure feedstocks  
#dfS1 = df.loc[df['Substrate'] != 'CO2_1:H2_4']
color_scale1 = [[0, 'red'], [0.5, 'lightblue'], [1, 'blue']]
color_scale2 = [[0, 'gold'], [0.5, 'lightgreen'], [1, 'purple']]
ce_threshold = 0.85

dfS1 = df[df["Substrate"].isin(['Glucose', ' Glycerol', 'Xylose', ' Methanol_Formolase', 'Methanol_RuMP', 'Methanol_SerineCycle', 
                                 'Formate_Formolase', 'Formate_RGP-Serine', 'Formate_SerineCycle', 'Acetaldehyde', 'Acetate', 
                                 'EG_Glycolate', ' EG_SACA', ' Glycolaldehyde_Glycolate', 'Glycolaldehyde_SACA','Methanol_MethylTransferase_1',
                                'Methanol_MethylTransferase_0.5','Methanol_MethylTransferase_0.1','Methanol_MethylTransferase_0'])]
#display(dfS1)

dfS2 = df[df["Substrate"].isin(["CO2_1:H2_3", "FOR_1:H2_0", "FOR_1:H2_0.5", "FOR_1:H2_1", "FOR_1:H2_2", "CO_1:H2_0", "CO_1:H2_0.5", "CO_1:H2_1", "CO_1:H2_2","CO_2:CO1_4"])]
# For Pure Feedstocks
dataS1 = go.Scatter(x=dfS1[X1], y=dfS1[Y], mode='markers', orientation=ORI,
                    marker=dict(color=dfS1['CE'], colorscale=color_scale1,colorbar=dict(title='Pure Feedstock, Carbon Efficiency', x=1, xanchor='left', titleside='right')),
                    marker_symbol='diamond', marker_size=markerSize, name='Pure Feedstocks', showlegend=True)

# For H2 Mixtures
dataS2 = go.Scatter(x=dfS2[X2], y=dfS2[Y], mode='markers', orientation=ORI,
                    marker=dict(color=dfS2['CE'], colorscale=color_scale2,colorbar=dict(title='H2 Mix, Carbon Efficiency', x=1.15, xanchor='left', titleside='right')),
                    marker_symbol='circle-open', marker_size=markerSize, name='H2 Mixtures', showlegend=True)

data = [dataS1,dataS2]


layout = go.Layout(
                   width=W, height=H, template="simple_white", plot_bgcolor = "white",
                   margin=go.layout.Margin(l=0, r=20,b=0, t=50),  legend=dict(x=1.04, y=-0.07,xanchor='left', yanchor='bottom',font=dict(family='sans-serif', size=16, color='black')),                  
                   )
fig = go.Figure(data = data, layout = layout)



if ORI == 'v':
        fig.update_xaxes(title_text = 'Degree of Reduction Ratio (S/P)', title_font_size= 18, tickfont_size=18,
                 showline=True, linewidth=1, color='black', mirror = True,
                 showgrid=True, gridcolor='#eee',
                 range=[0,3.5])
        fig.update_yaxes(title_text = 'Mass Yield (g/g)', title_font_size= 18, tickfont_size=18,
                 showline=True, linewidth=1, color='black', mirror = True, showgrid=True, gridcolor='#eee',
                 range=[0,2.5])
        
    
    
#fig.update_layout(legend=dict(yanchor="bottom", y=0.01, xanchor="right", x=0.99))

fig.show()
fig.write_image(outputDirectory_general + date + "_DoR-Ratios_" + ".png", scale=5)

<>:3: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:3: SyntaxWarning:

"is" with a literal. Did you mean "=="?

C:\Users\austi\AppData\Local\Temp\ipykernel_23548\2253388590.py:3: SyntaxWarning:

"is" with a literal. Did you mean "=="?



## Energy Cost Analysis ##

In [ ]:
scenario = 'Categories' #Ideal or Realistic
case = 'Category_2' #Worst, Base, Best
FileName = '2024-05-08_Type-Bio_Case-'+ case +'_Analysis-' + scenario + '_FULL.xlsx'
SheetName = 'Overall-ECR-Bio'
Type = 'ECR-Bio'

df = readInData(FileName, SheetName, Type)
n = 57

df.loc[df['SubstratePathway'] == 'FOR_1:H2_0', 'SubstrateType'] = 'C1+H2'
df.loc[df['SubstratePathway'] == 'CO_1:H2_0', 'SubstrateType'] = 'C1+H2'


#To graph only a subset of bioproducts:
#df = df[(df['BioProduct'].isin(['Propane-PW1','Butanol-PW2', '1-3-Diaminopropane', 'Isoprene', '2-3-Butanediol','Lysine_L', 'Catechol', 'Adipate', 'Succinate', 'Malonate']))]
#n = 10

display(df)

In [53]:
##----- Bioproduction Energy Efficiency (Single Pass/Recycle) -----##

#df = df.loc[df['MoleculeType_P'] == 'Hydrocarbon (alkane)']
#df = df.sort_values(by='C_P', ascending=0)
df = df.sort_values(by='Deg_RedBP', ascending=0)
df = df[df['Deg_RedBP'] <= 5.601] # For sorting chems/fuels
#display(df)

ORI = 'h' #'v' for vertical, 'h' for horizontal
S = 'SubstratePathway'

# Potential X/Y axis: Carbon_capture_SP, Carbon_capture_rec, Flue_gas_SP, Flue_gas_rec, Bioreactor_SP, Bioreactor_rec
if ORI is 'v':
    X = 'BioProduct_Graph'
    Y = 'Carbon_capture_rec'
    W = 1400
    H = 800
    title_pos = 0.45
    #For glucose costs:
    X2 = X
    if case is 'Category_2':
        Y2 = 'Cost_Best'
    elif case is 'Category_1':
        Y2 = 'Cost_Base'    
elif ORI is 'h':
    X = 'Carbon_capture_rec'
    Y = 'BioProduct_Graph'
    W = 900
    H = 900
    title_pos = 0.6
    #For glucose costs:
    Y2 = Y
    if case is 'Category_2':
        X2 = 'Cost_Best'
    elif case is 'Category_1':
        X2 = 'Cost_Base'

markerSize=7

fig = go.Figure()

dfS1 = df2.loc[df2[S] == 'Glucose']
dataS1 = go.Scatter(x=dfS1[X2], y=dfS1[Y2], mode='markers', orientation=ORI,
                    marker_color= 'gold', marker_line_color='black', marker_line_width=0.75, marker_symbol= 'diamond', marker_size=markerSize,
                    name = 'Glucose'
                    )

dfS2 = df.loc[df[S] == 'Glycerol']
dataS2 = go.Scatter(x=dfS2[X], y=dfS2[Y], mode='markers', orientation=ORI,
                    marker_color= 'red',marker_symbol= 'diamond', marker_size=markerSize,
                    name = 'Glycerol'
                    )

dfS3 = df.loc[df[S] == 'Xylose'] 
dataS3 = go.Scatter(x=dfS3[X], y=dfS3[Y], mode='markers', orientation=ORI,
                    marker_color= 'blue', marker_symbol= 'diamond', marker_size=markerSize,
                    name = 'Xylose'
                    )

#ECR 1-C compounds; iJO1366
dfC11 = df.loc[df[S] == 'Methanol_Formolase']
dataC11 = go.Scatter(x=dfC11[X], y=dfC11[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkviolet', marker_symbol= 'square', marker_size = markerSize+1,
                    name = 'MeOH - Formolase',    
                    )

dfC12 = df.loc[df[S] == 'Methanol_RuMP']
dataC12 = go.Scatter(x=dfC12[X], y=dfC12[Y], mode='markers', orientation=ORI,
                    marker_color= 'deepskyblue', marker_symbol= 'square', marker_size = markerSize-1,
                    name = 'MeOH - RuMP',
                    )

dfC13 = df.loc[df[S] == 'Methanol_SerineCycle']
dataC13 = go.Scatter(x=dfC13[X], y=dfC13[Y], mode='markers', orientation=ORI,
                    marker_color= 'red', marker_symbol= 'square', marker_size = markerSize,
                    name = 'MeOH - SerineCycle'    
                    )

dfC14 = df.loc[df[S] == 'Formate_Formolase']
dataC14 = go.Scatter(x=dfC14[X], y=dfC14[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkviolet', marker_symbol= 'x', marker_size = markerSize,
                    name = 'Formate - Formolase',
                    )

dfC15 = df.loc[df[S] == 'Formate_RGP-Serine']
dataC15 = go.Scatter(x=dfC15[X], y=dfC15[Y],mode='markers', orientation=ORI,
                    marker_color= 'olivedrab', marker_symbol= 'x', marker_size = markerSize,
                    name = 'Formate - RGP',
                    )

dfC16 = df.loc[df[S] == 'Formate_SerineCycle']
dataC16 = go.Scatter(x=dfC16[X], y=dfC16[Y],mode='markers', orientation=ORI,
                    marker_color= 'red', marker_symbol= 'x', marker_size = markerSize,
                    name = 'Formate - Serine Cycle',
                    )

#ECR 2-C compounds; iJO1366
dfC21 = df.loc[df[S] == 'Acetaldehyde']
dataC21 = go.Scatter(x=dfC21[X], y=dfC21[Y], mode='markers', orientation=ORI,
                    marker_color='darkblue', marker_symbol= 'circle', marker_size =markerSize,
                    name = 'Acetaldehyde',
                    )

dfC22 = df.loc[df[S] == 'Acetate']
dataC22 = go.Scatter(x=dfC22[X], y=dfC22[Y], mode='markers', orientation=ORI,
                    marker_color= 'red', marker_symbol= 'circle', marker_size =markerSize,
                    name = 'Acetate',
                    )


dfC23 = df.loc[df[S] == 'Ethanol']
dataC23 = go.Scatter(x=dfC23[X], y=dfC23[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkorange',marker_symbol= 'circle', marker_size =markerSize,
                    name = 'Ethanol',
                    )

dfC24 = df.loc[df[S] == 'EG_Glycolate']
dataC24 = go.Scatter(x=dfC24[X], y=dfC24[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkblue', marker_symbol= 'triangle-up', marker_size =markerSize,
                    name = 'EG - Glycolate',
                    )

dfC25 = df.loc[df[S] == 'EG_SACA']
dataC25 = go.Scatter(x=dfC25[X], y=dfC25[Y],mode='markers', orientation=ORI,
                    marker_color= 'red', marker_symbol= 'triangle-up', marker_size =markerSize,
                    name = 'EG - SACA',
                    )



dfC26 = df.loc[df[S] == 'Glycolaldehyde_Glycolate']
dataC26 = go.Scatter(x=dfC26[X], y=dfC26[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkorange', marker_symbol= 'triangle-up', marker_size =markerSize,
                    name = 'GlyAld - Glycolate',
                    )

dfC27 = df.loc[df[S] == 'Glycolaldehyde_SACA']
dataC27 = go.Scatter(x=dfC27[X], y=dfC27[Y], mode='markers', orientation=ORI,
                    marker_color= 'deepskyblue', marker_symbol= 'triangle-up', marker_size =markerSize,
                    name = 'GlyAld - SACA',
                    )
#ECR 3-C compounds; iJO1366
dfC31 = df.loc[df[S] == 'N_Propanol']
dataC31 = go.Scatter(x=(dfC31[X]*3/2), y=dfC31[Y], mode='markers', orientation=ORI,
                    marker_color= 'black', marker_symbol= 'circle', marker_size =markerSize,
                    name = 'n-Propanol',
                    ) # The *(3/2) modification is necessary for the carbon efficiency as for now I've assumed propanol as "ethanol"

#ECR - iHN637 - CO + Formate Mixes
dfH1 = df.loc[df[S] == 'FOR_1:H2_0'] 
dataH1 = go.Scatter(x=dfH1[X], y=dfH1[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkgreen', marker_symbol= 'star-square-open', marker_size =markerSize,
                    name = '1 FOR:0 H2',
                    )

dfH2 = df.loc[df[S] == 'FOR_1:H2_0.5']
dataH2 = go.Scatter(x=dfH2[X], y=dfH2[Y], mode='markers', orientation=ORI,
                    marker_color= 'red', marker_symbol= 'star-square-open', marker_size =markerSize,
                    name = '1 FOR:0.5 H2',
                    )

dfH3 = df.loc[df[S] == 'FOR_1:H2_1']
dataH3 = go.Scatter(x=dfH3[X], y=dfH3[Y], mode='markers', orientation=ORI,
                    marker_color= 'dodgerblue', marker_symbol= 'star-square-open', marker_size =markerSize,
                    name = '1 FOR:1 H2',
                    )

dfH4 = df.loc[df[S] == 'FOR_1:H2_2']
dataH4 = go.Scatter(x=dfH4[X], y=dfH4[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkviolet', marker_symbol= 'star-square-open', marker_size =markerSize,
                    name = '1 FOR:2 H2',
                    )

dfH5 = df.loc[df[S] == 'CO_1:H2_0']
dataH5 = go.Scatter(x=dfH5[X], y=dfH5[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkgreen', marker_symbol= 'diamond-tall-open', marker_size =markerSize+0.5,
                    name = '1 CO:0 H2',
                    )

dfH6 = df.loc[df[S] == 'CO_1:H2_0.5'] 
dataH6 = go.Scatter(x=dfH6[X], y=dfH6[Y], mode='markers', orientation=ORI,
                    marker_color= 'red', marker_symbol= 'diamond-tall-open', marker_size =markerSize+0.5,
                    name = '1 CO:0.5 H2',
                    )

dfH7 = df.loc[df[S] == 'CO_1:H2_1']
dataH7 = go.Scatter(x=dfH7[X], y=dfH7[Y], mode='markers', orientation=ORI,
                    marker_color= 'dodgerblue', marker_symbol= 'diamond-tall-open', marker_size =markerSize+0.5,
                    name = '1 CO:1 H2',
                    )

dfH8 = df.loc[df[S] == 'CO_1:H2_2']
dataH8 = go.Scatter(x=dfH8[X], y=dfH8[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkviolet', marker_symbol= 'diamond-tall-open',  marker_size =markerSize+0.5,
                    name = '1 CO:2 H2',
                    )

dfH9 = df.loc[df[S] == 'CO_2:H2_3']
dataH9 = go.Scatter(x=dfH9[X], y=dfH9[Y], mode='markers', orientation=ORI,
                    marker_color= 'black', marker_symbol= 'circle-open',  marker_size =markerSize,
                    name = '1 CO2:X H2',
                    )

dfH14 = df.loc[df[S] == 'CO_2:CO1_4']
dataH14 = go.Scatter(x=dfH14[X], y=dfH14[Y], mode='markers', orientation=ORI,
                    marker_color= 'black', marker_symbol= 'diamond-tall-open',  marker_size =markerSize,
                    name = '1 CO2:4 CO',
                    )

dfH15 = df.loc[df[S] == 'Flue_Gas']
dataH15 = go.Scatter(x=dfH15[X], y=dfH15[Y], mode='markers', orientation=ORI,
                    marker_color= 'black', marker_symbol= 'star-square-open',  marker_size =markerSize,
                    name = 'Flue Gas',
                    )


#ECR - iHN637 - Methanogensis

dfH10 = df.loc[df[S] == 'Methanol_MethylTransferase_1']
dataH10 = go.Scatter(x=dfH10[X], y=dfH10[Y], mode='markers', orientation=ORI,
                    marker_color= 'dodgerblue', marker_symbol= 'star-open', marker_size = markerSize+4,
                    name = 'Methanol Fixation 1 ATP',    
                    )

dfH11 = df.loc[df[S] == 'Methanol_MethylTransferase_0.5']
dataH11 = go.Scatter(x=dfH11[X], y=dfH11[Y], mode='markers', orientation=ORI,
                    marker_color= 'red', marker_symbol= 'star-open', marker_size = markerSize+3,
                    name = 'Methanol Fixation 0.5 ATP',    
                    )

dfH12 = df.loc[df[S] == 'Methanol_MethylTransferase_0.1']
dataH12 = go.Scatter(x=dfH12[X], y=dfH12[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkgreen', marker_symbol= 'star-open', marker_size = markerSize+2,
                    name = 'Methanol Fixation 0.1 ATP',    
                    )

dfH13 = df.loc[df[S] == 'Methanol_MethylTransferase_0']
dataH13 = go.Scatter(x=dfH13[X], y=dfH13[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkviolet', marker_symbol= 'star-open', marker_size = markerSize+1,
                    name = 'Methanol Fixation no ATP',    
                    )
                    

data = [dataC21, dataC22, dataC23, dataC24, dataC25, dataC26, dataC27,
        dataC11, dataC12, dataC13, dataC14, dataC15, dataC16, dataC31,
        dataH1, dataH2, dataH3, dataH4, dataH5, dataH6, dataH7, dataH8, dataH9,
        dataS2, dataS3, dataH10, dataH11, dataH12, dataH13, dataH14 #dataH15
        ]

if ORI is 'v':
    shape1=dict(type="line", x0=-1, x1=57, y0=1, y1=1, xref='x', yref='y', line=dict(color='black', width=1.5))
elif ORI is 'h':
    shape1=dict(type="line", x0=1, x1=1, y0=-1, y1=57, xref='x', yref='y', line=dict(color='black', width=1.5))
#shape2=dict(type="rect", x0=6, x1=10, y0=0, y1=2.5, xref='x', yref='y',fillcolor='orange', opacity=0.2, line=dict(color='darkorange', width=2))


shapes = [shape1]

layout = go.Layout(title_text='Energy Requirements (DAC Rec - Chemicals)', title = {'x': title_pos},
                   width=W, height=H, template="simple_white", plot_bgcolor = "white",
                   margin=go.layout.Margin(l=0, r=20,b=0, t=50),
                  legend_font_size=14.5, #20 #for ORI=h,#)
                  )

fig = go.Figure(data = data, layout = layout)



if ORI == 'v':
        fig.update_xaxes(title_text = 'Bioproducts', title_font_size= 16, tickfont_size=14, tickangle=-90,
                 showline=True, linewidth=1, color='black', mirror = True,
                 showgrid=True, gridcolor='#eee',
                 range=[-1,57])
        fig.update_yaxes(title_text = 'Energy Requirements (GJ/tonne)', title_font_size= 16, tickfont_size=14,
                 showline=True, linewidth=1, color='black', mirror = True, showgrid=True, gridcolor='#eee',
                 range=[0,2.5])
        
    
    
elif ORI == 'h':

    fig.update_xaxes(title_text = 'Energy Requirements (GJ/tonne)', title_font_size= 20, tickfont_size=20,
                 showline=True, linewidth=1, color='black', mirror = True, showgrid=True, gridcolor='#eee',
                range=[20,65], tick0 = 0)
    fig.update_yaxes(title_text = 'Bioproducts', title_font_size= 20, tickfont_size=15,
                 showline=True, linewidth=1, color='black', mirror = True,
                 showgrid=True, gridcolor='#eee',
                 range=[-1,38]) 

    fig.update_layout(legend=dict(yanchor="bottom", y=0.01, xanchor="right", x=0.99),showlegend = False)

fig.show()
fig.write_image(outputDirectory_general + date + "_EnergyInput_DAC_Rec_Chem" + ORI + ".png", scale=5)

<>:13: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:21: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:23: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:25: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:33: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:35: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:250: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:252: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:13: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:21: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:23: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:25: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:33: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:35: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:250: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:252: SyntaxWarning:

"is" with a l

In [9]:
##----- Bioproduction Energy Input  -----##

#df = df.loc[df['MoleculeType_P'] == 'Hydrocarbon (alkane)']
#df = df.sort_values(by='C_P', ascending=0)
df = df.sort_values(by='Deg_RedBP', ascending=0)
#display(df)

ORI = 'h' #'v' for vertical, 'h' for horizontal
S = 'SubstratePathway'

if ORI is 'v':
    X = 'BioProduct_Graph'
    Y = 'Energy_Input'
    W = 1400
    H = 800
    title_pos = 0.45
    #For glucose costs:
    X2 = X
    if case is 'Category_2':
        Y2 = 'Cost_Best'
    elif case is 'Category_1':
        Y2 = 'Cost_Base'    
elif ORI is 'h':
    X = 'Energy_Input'
    Y = 'BioProduct_Graph'
    W = 1200
    H = 1500
    title_pos = 0.6
    #For glucose costs:
    Y2 = Y
    if case is 'Category_2':
        X2 = 'Cost_Best'
    elif case is 'Category_1':
        X2 = 'Cost_Base'

markerSize=7

fig = go.Figure()

#dfS1 = df2.loc[df2[S] == 'Glucose']
#dataS1 = go.Scatter(x=dfS1[X2], y=dfS1[Y2], mode='markers', orientation=ORI,
#                    marker_color= 'gold', marker_line_color='black', marker_line_width=0.75, marker_symbol= 'diamond', marker_size=markerSize,
#                    name = 'Glucose'
#                    )
#No Glucose Cost for now

dfS2 = df.loc[df[S] == 'Glycerol']
dataS2 = go.Scatter(x=dfS2[X], y=dfS2[Y], mode='markers', orientation=ORI,
                    marker_color= 'red',marker_symbol= 'diamond', marker_size=markerSize,
                    name = 'Glycerol'
                    )

dfS3 = df.loc[df[S] == 'Xylose'] 
dataS3 = go.Scatter(x=dfS3[X], y=dfS3[Y], mode='markers', orientation=ORI,
                    marker_color= 'blue', marker_symbol= 'diamond', marker_size=markerSize,
                    name = 'Xylose'
                    )

#ECR 1-C compounds; iJO1366
dfC11 = df.loc[df[S] == 'Methanol_Formolase']
dataC11 = go.Scatter(x=dfC11[X], y=dfC11[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkviolet', marker_symbol= 'square', marker_size = markerSize+1,
                    name = 'MeOH - Formolase',    
                    )

dfC12 = df.loc[df[S] == 'Methanol_RuMP']
dataC12 = go.Scatter(x=dfC12[X], y=dfC12[Y], mode='markers', orientation=ORI,
                    marker_color= 'deepskyblue', marker_symbol= 'square', marker_size = markerSize-1,
                    name = 'MeOH - RuMP',
                    )

dfC13 = df.loc[df[S] == 'Methanol_SerineCycle']
dataC13 = go.Scatter(x=dfC13[X], y=dfC13[Y], mode='markers', orientation=ORI,
                    marker_color= 'red', marker_symbol= 'square', marker_size = markerSize,
                    name = 'MeOH - SerineCycle'    
                    )

dfC14 = df.loc[df[S] == 'Formate_Formolase']
dataC14 = go.Scatter(x=dfC14[X], y=dfC14[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkviolet', marker_symbol= 'x', marker_size = markerSize,
                    name = 'Formate - Formolase',
                    )

dfC15 = df.loc[df[S] == 'Formate_RGP-Serine']
dataC15 = go.Scatter(x=dfC15[X], y=dfC15[Y],mode='markers', orientation=ORI,
                    marker_color= 'olivedrab', marker_symbol= 'x', marker_size = markerSize,
                    name = 'Formate - RGP',
                    )

dfC16 = df.loc[df[S] == 'Formate_SerineCycle']
dataC16 = go.Scatter(x=dfC16[X], y=dfC16[Y],mode='markers', orientation=ORI,
                    marker_color= 'red', marker_symbol= 'x', marker_size = markerSize,
                    name = 'Formate - Serine Cycle',
                    )

#ECR 2-C compounds; iJO1366
dfC21 = df.loc[df[S] == 'Acetaldehyde']
dataC21 = go.Scatter(x=dfC21[X], y=dfC21[Y], mode='markers', orientation=ORI,
                    marker_color='darkblue', marker_symbol= 'circle', marker_size =markerSize,
                    name = 'Acetaldehyde',
                    )

dfC22 = df.loc[df[S] == 'Acetate']
dataC22 = go.Scatter(x=dfC22[X], y=dfC22[Y], mode='markers', orientation=ORI,
                    marker_color= 'red', marker_symbol= 'circle', marker_size =markerSize,
                    name = 'Acetate',
                    )


dfC23 = df.loc[df[S] == 'Ethanol']
dataC23 = go.Scatter(x=dfC23[X], y=dfC23[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkorange',marker_symbol= 'circle', marker_size =markerSize,
                    name = 'Ethanol',
                    )

dfC24 = df.loc[df[S] == 'EG_Glycolate']
dataC24 = go.Scatter(x=dfC24[X], y=dfC24[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkblue', marker_symbol= 'triangle-up', marker_size =markerSize,
                    name = 'EG - Glycolate',
                    )

dfC25 = df.loc[df[S] == 'EG_SACA']
dataC25 = go.Scatter(x=dfC25[X], y=dfC25[Y],mode='markers', orientation=ORI,
                    marker_color= 'red', marker_symbol= 'triangle-up', marker_size =markerSize,
                    name = 'EG - SACA',
                    )



dfC26 = df.loc[df[S] == 'Glycolaldehyde_Glycolate']
dataC26 = go.Scatter(x=dfC26[X], y=dfC26[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkorange', marker_symbol= 'triangle-up', marker_size =markerSize,
                    name = 'GlyAld - Glycolate',
                    )

dfC27 = df.loc[df[S] == 'Glycolaldehyde_SACA']
dataC27 = go.Scatter(x=dfC27[X], y=dfC27[Y], mode='markers', orientation=ORI,
                    marker_color= 'deepskyblue', marker_symbol= 'triangle-up', marker_size =markerSize,
                    name = 'GlyAld - SACA',
                    )
#ECR 3-C compounds; iJO1366
dfC31 = df.loc[df[S] == 'N_Propanol']
dataC31 = go.Scatter(x=(dfC31[X]*3/2), y=dfC31[Y], mode='markers', orientation=ORI,
                    marker_color= 'black', marker_symbol= 'circle', marker_size =markerSize,
                    name = 'n-Propanol',
                    ) # The *(3/2) modification is necessary for the carbon efficiency as for now I've assumed propanol as "ethanol" carbon eff, thus in the calculations it assumes 2 mols C instead of 3

#ECR - iHN637 - CO + Formate Mixes
dfH1 = df.loc[df[S] == 'FOR_1:H2_0'] 
dataH1 = go.Scatter(x=dfH1[X], y=dfH1[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkgreen', marker_symbol= 'star-square-open', marker_size =markerSize,
                    name = '1 FOR:0 H2',
                    )

dfH2 = df.loc[df[S] == 'FOR_1:H2_0.5']
dataH2 = go.Scatter(x=dfH2[X], y=dfH2[Y], mode='markers', orientation=ORI,
                    marker_color= 'red', marker_symbol= 'star-square-open', marker_size =markerSize,
                    name = '1 FOR:0.5 H2',
                    )

dfH3 = df.loc[df[S] == 'FOR_1:H2_1']
dataH3 = go.Scatter(x=dfH3[X], y=dfH3[Y], mode='markers', orientation=ORI,
                    marker_color= 'dodgerblue', marker_symbol= 'star-square-open', marker_size =markerSize,
                    name = '1 FOR:1 H2',
                    )

dfH4 = df.loc[df[S] == 'FOR_1:H2_2']
dataH4 = go.Scatter(x=dfH4[X], y=dfH4[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkviolet', marker_symbol= 'star-square-open', marker_size =markerSize,
                    name = '1 FOR:2 H2',
                    )

dfH5 = df.loc[df[S] == 'CO_1:H2_0']
dataH5 = go.Scatter(x=dfH5[X], y=dfH5[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkgreen', marker_symbol= 'diamond-tall-open', marker_size =markerSize+0.5,
                    name = '1 CO:0 H2',
                    )

dfH6 = df.loc[df[S] == 'CO_1:H2_0.5'] 
dataH6 = go.Scatter(x=dfH6[X], y=dfH6[Y], mode='markers', orientation=ORI,
                    marker_color= 'red', marker_symbol= 'diamond-tall-open', marker_size =markerSize+0.5,
                    name = '1 CO:0.5 H2',
                    )

dfH7 = df.loc[df[S] == 'CO_1:H2_1']
dataH7 = go.Scatter(x=dfH7[X], y=dfH7[Y], mode='markers', orientation=ORI,
                    marker_color= 'dodgerblue', marker_symbol= 'diamond-tall-open', marker_size =markerSize+0.5,
                    name = '1 CO:1 H2',
                    )

dfH8 = df.loc[df[S] == 'CO_1:H2_2']
dataH8 = go.Scatter(x=dfH8[X], y=dfH8[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkviolet', marker_symbol= 'diamond-tall-open',  marker_size =markerSize+0.5,
                    name = '1 CO:2 H2',
                    )

dfH9 = df.loc[df[S] == 'CO_2:H2_3']
dataH9 = go.Scatter(x=dfH9[X], y=dfH9[Y], mode='markers', orientation=ORI,
                    marker_color= 'black', marker_symbol= 'circle-open',  marker_size =markerSize,
                    name = '1 CO2:X H2',
                    )

dfH14 = df.loc[df[S] == 'CO_2:CO1_4']
dataH14 = go.Scatter(x=dfH14[X], y=dfH14[Y], mode='markers', orientation=ORI,
                    marker_color= 'black', marker_symbol= 'diamond-tall-open',  marker_size =markerSize,
                    name = '1 CO2:4 CO',
                    )

dfH15 = df.loc[df[S] == 'Flue_Gas']
dataH15 = go.Scatter(x=dfH15[X], y=dfH15[Y], mode='markers', orientation=ORI,
                    marker_color= 'black', marker_symbol= 'star-square-open',  marker_size =markerSize,
                    name = 'Flue Gas',
                    )


#ECR - iHN637 - Methanogensis

dfH10 = df.loc[df[S] == 'Methanol_MethylTransferase_1']
dataH10 = go.Scatter(x=dfH10[X], y=dfH10[Y], mode='markers', orientation=ORI,
                    marker_color= 'dodgerblue', marker_symbol= 'star-open', marker_size = markerSize+4,
                    name = 'Methanol Fixation 1 ATP',    
                    )

dfH11 = df.loc[df[S] == 'Methanol_MethylTransferase_0.5']
dataH11 = go.Scatter(x=dfH11[X], y=dfH11[Y], mode='markers', orientation=ORI,
                    marker_color= 'red', marker_symbol= 'star-open', marker_size = markerSize+3,
                    name = 'Methanol Fixation 0.5 ATP',    
                    )

dfH12 = df.loc[df[S] == 'Methanol_MethylTransferase_0.1']
dataH12 = go.Scatter(x=dfH12[X], y=dfH12[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkgreen', marker_symbol= 'star-open', marker_size = markerSize+2,
                    name = 'Methanol Fixation 0.1 ATP',    
                    )

dfH13 = df.loc[df[S] == 'Methanol_MethylTransferase_0']
dataH13 = go.Scatter(x=dfH13[X], y=dfH13[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkviolet', marker_symbol= 'star-open', marker_size = markerSize+1,
                    name = 'Methanol Fixation no ATP',    
                    )
                    

data = [dataC21, dataC22, dataC23, dataC24, dataC25, dataC26, dataC27,
        dataC11, dataC12, dataC13, dataC14, dataC15, dataC16, dataC31,
        dataH1, dataH2, dataH3, dataH4, dataH5, dataH6, dataH7, dataH8, dataH9,
        dataS2, dataS3, dataH10, dataH11, dataH12, dataH13, dataH14,] # dataH15 - Flue Gas removed
#data = [dataC11, dataC12, dataC13]

#if ORI is 'v':
#    shape1=dict(type="line", x0=-1, x1=57, y0=1, y1=1, xref='x', yref='y', line=dict(color='black', width=1.5))
#elif ORI is 'h':
#    shape1=dict(type="line", x0=1, x1=1, y0=-1, y1=57, xref='x', yref='y', line=dict(color='black', width=1.5))
#shape2=dict(type="rect", x0=6, x1=10, y0=0, y1=2.5, xref='x', yref='y',fillcolor='orange', opacity=0.2, line=dict(color='darkorange', width=2))


#shapes = [shape1]

layout = go.Layout(title_text='Bioproduction Energy Input', title = {'x': title_pos},
                   width=W, height=H, template="simple_white", plot_bgcolor = "white",
                   margin=go.layout.Margin(l=0, r=20,b=0, t=50),
                  legend_font_size=14.5, #20 #for ORI=h,#)
                  #shapes=shapes)
)

fig = go.Figure(data = data, layout = layout)



if ORI == 'v':
        fig.update_xaxes(title_text = 'Bioproducts', title_font_size= 16, tickfont_size=14, tickangle=-90,
                 showline=True, linewidth=1, color='black', mirror = True,
                 showgrid=True, gridcolor='#eee',
                 range=[-1,57])
        fig.update_yaxes(title_text = 'Energy Requirements (GJ/tonne)', title_font_size= 16, tickfont_size=14,
                 showline=True, linewidth=1, color='black', mirror = True, showgrid=True, gridcolor='#eee',
                 range=[0,150])
        
    
    
elif ORI == 'h':

    fig.update_xaxes(title_text = 'Energy Requirements (GJ/tonne)', title_font_size= 20, tickfont_size=20,
                 showline=True, linewidth=1, color='black', mirror = True, showgrid=True, gridcolor='#eee',
                range=[0,150], tick0 = 0)
    fig.update_yaxes(title_text = 'Bioproducts', title_font_size= 20, tickfont_size=15,
                 showline=True, linewidth=1, color='black', mirror = True,
                 showgrid=True, gridcolor='#eee',
                 range=[-1,57]) 

    fig.update_layout(legend=dict(yanchor="bottom", y=0.01, xanchor="right", x=0.99))

fig.show()
fig.write_image(outputDirectory_general + date + "_BioproductionEnergy_Input" + ORI + ".png", scale=5)
fig.write_html(outputDirectory_general + date + "_BioproductionEnergy_Input" + ORI + ".html")

<>:11: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:19: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:21: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:23: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:31: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:33: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:11: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:19: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:21: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:23: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:31: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:33: SyntaxWarning:

"is" with a literal. Did you mean "=="?

C:\Users\austi\AppData\Local\Temp\ipykernel_16404\2115265666.py:11: SyntaxWarning:

"is" with a literal. Did you mean "=="?

C:\Users\austi\AppData\Local\Temp\ipykernel_16404\2115265666.py:19: SyntaxWarning:

"is" with a literal. Di

In [95]:
## ----- SI Yield Graph ----- ##

# Read the Excel file into a pandas DataFrame
df = pd.read_excel('SIYield.xlsx', sheet_name='Sheet1')

# Remove the duplicate methanol pathways and propanol and flue gas cases due to incorrect SI yields (due to earlier assumptions). These are fixed in the final results and previous figures
# but cannot be easily accounted for for the purposes of this yield figure (although after adjustment they are not in the top performing intermediates)
df_filtered = df[~df['SubstratePathway'].isin(['N_Propanol', 'Flue_Gas','Methanol_MethylTransferase_1','Methanol_MethylTransferase_0.5'])]

# Group the DataFrame by 'BioProduct_Graph' and find the top 4 yields within each group
top_yields_per_bioproduct = df_filtered.groupby('BioProduct_Graph', as_index=False).apply(lambda x: x.nlargest(5, 'YieldBP_Mass'))

# Calculate the mean of the top 4 yields for each BioProduct
mean_top_yields_per_bioproduct = top_yields_per_bioproduct.groupby(['BioProduct_Graph', 'Deg_RedBP'])['YieldBP_Mass'].mean().reset_index()

# Fuels or Chemicals
mean_top_yields_per_bioproduct = mean_top_yields_per_bioproduct[mean_top_yields_per_bioproduct['Deg_RedBP'] > 5.5]

mean_top_yields_per_bioproduct.sort_values(by='YieldBP_Mass', ascending=False, inplace=True)

print(mean_top_yields_per_bioproduct)
# Export the DataFrame to an Excel file
# mean_top_yields_per_bioproduct.to_excel('meandata.xlsx', index=False)

# Sort the DataFrame by descending means and take top 15
top_15_bioproducts = mean_top_yields_per_bioproduct.nlargest(15, 'YieldBP_Mass')

# Create a bar plot using Plotly
fig = go.Figure(go.Bar(
    x=top_15_bioproducts['BioProduct_Graph'],
    y=top_15_bioproducts['YieldBP_Mass'],
    marker_color='rgb(26, 118, 255)'
))

# Update layout
title_pos = 0.5
W, H = 800, 600

fig.update_layout(
    title_text='Best Mean Yields for Fuels',
    title={'x': title_pos},
    width=W,
    height=H,
    template="simple_white",
    plot_bgcolor="white",
    margin=go.layout.Margin(l=0, r=20, b=0, t=50),
    legend_font_size=14.5,
    xaxis_title='BioProduct',
    yaxis_title='Mean Yield'
)


# Show the plot
fig.show()

       BioProduct_Graph  Deg_RedBP  YieldBP_Mass
2    1,3-Diaminopropane   6.050000      0.778946
35           Isobutanol   5.602000      0.726582
38          Isopropanol   6.040000      0.673787
22        Butanol (PW1)   6.030000      0.640743
6      1-Propanol (PW2)   6.010000      0.594632
5      1-Propanol (PW1)   6.010000      0.587489
23        Butanol (PW2)   6.030000      0.587224
48       Pentanol (PW1)   6.060000      0.536649
8    2-Methyl-1-Butanol   6.020000      0.535527
10  3-Methyl-1-Pentanol   6.020000      0.478388
49       Pentanol (PW2)   6.000000      0.474984
45               Nonane   6.222222      0.469203
33              Heptane   6.285714      0.465130
46               Octane   6.250000      0.464157
34               Hexane   6.333333      0.458079
47              Pentane   6.400000      0.457199
51        Propane (PW1)   6.666667      0.441644
43             Limonene   5.600000      0.423674
29            Farnesene   5.601000      0.423171
37             Isopr

In [99]:
## ----- SI Yield Graph (Intermediates) ----- ##


# Read the Excel file into a pandas DataFrame
df = pd.read_excel('TestSI.xlsx', sheet_name='Sheet1')

# Remove BioProducts with Deg_RedBP > 5.5 (Fuels)
df_filtered = df[~df['BioProduct_Graph'].isin(df[df['Deg_RedBP'] > 5.5]['BioProduct_Graph'])]

# Remove entries with SubstratePathway = N_Propanol and Flue_Gas (Due to issues with their yield calculations that are fixed for overall efficiency figures but are difficult to adjust 
# for solely bioproduction mass yield). They do not fall within the best performing intermediates even after adjustment
df_filtered = df_filtered[~df_filtered['SubstratePathway'].isin(['N_Propanol', 'Flue_Gas'])]

# Group the DataFrame by 'SubstratePathway' and find the top 10 yields within each group
top_yields_per_pathway = df_filtered.groupby('SubstratePathway', as_index=False).apply(lambda x: x.nlargest(10, 'YieldBP_Mass'))

# Calculate the mean of the top 4 yields for each SubstratePathway
mean_top_yields_per_pathway = top_yields_per_pathway.groupby('SubstratePathway')['YieldBP_Mass'].mean().reset_index()

# Sort the DataFrame by descending means and take top 15
top_15_pathways = mean_top_yields_per_pathway.nlargest(15, 'YieldBP_Mass')

# Create a bar plot using Plotly
fig = go.Figure(go.Bar(
    x=top_15_pathways['SubstratePathway'],
    y=top_15_pathways['YieldBP_Mass'],
    marker_color='rgb(26, 118, 255)'
))

# Update layout
title_pos = 0.5
W, H = 800, 600

fig.update_layout(
    title_text='Best Mean Yields for Intermediates (Chemicals)',
    title={'x': title_pos},
    width=W,
    height=H,
    template="simple_white",
    plot_bgcolor="white",
    margin=go.layout.Margin(l=0, r=20, b=0, t=50),
    legend_font_size=14.5,
    xaxis_title='ECR Intermediate_Pathway',
    yaxis_title='Mean Yield'
)

# Show the plot
fig.show()


In [9]:
## ----- Discussion Figure, Recycle vs SP ----- ##

scenario = 'Categories' #Ideal or Realistic
case = 'Category_2' #Worst, Base, Best
FileName = '2024-05-08_Type-Bio_Case-'+ case +'_Analysis-' + scenario + '_FULL.xlsx'
SheetName = 'Overall-ECR-Bio'
Type = 'ECR-Bio'

df = readInData(FileName, SheetName, Type)

# Pick a product-substrate pairing to plot
filtered_df = df[(df['BioProduct'] == 'Isobutanol') & (df['SubstratePathway'] == 'Glycolaldehyde_SACA')]

coinput_values = filtered_df[['CCSP_COInput', 'BioSP_COInput', 'CCRec_COInput', 'BioRec_COInput']].values[0]

#display(co_input_values)
cobio_values = filtered_df[['CCSP_COBio', 'BioSP_COBio', 'CCRec_COBio', 'BioRec_COBio']].values[0]
rec_values = filtered_df[['CCSP_GasRec', 'BioSP_GasRec', 'CCRec_Rec', 'BioRec_Rec']].values[0]
ecr_values = filtered_df[['CCSP_ECR', 'BioSP_ECR', 'CCRec_ECR', 'BioRec_ECR']].values[0]


# Step 4: Create Plotly stacked bar plot
fig = go.Figure(data=[
    go.Bar(name='ECR Energy Input', x=['DAC_SP', 'Bio_SP', 'DAC_Rec', 'Bio_Rec'], y=ecr_values),
    go.Bar(name='CO<sub>2</sub> Input', x=['DAC_SP', 'Bio_SP', 'DAC_Rec', 'Bio_Rec'], y=coinput_values),
    #go.Bar(name='CO2 Bioreactor', x=['DAC_SP', 'Bio_SP', 'DAC_Rec', 'Bio_Rec'], y=cobio_values),
    go.Bar(name='Recycle/Separation', x=['DAC_SP', 'Bio_SP', 'DAC_Rec', 'Bio_Rec'], y=rec_values, marker=dict(color=['#D62728', '#D62728', '#D62728', '#D62728']))
    
])

# Update layout
fig.update_layout(
    title='Isobutanol Production From Glycoaldehyde (SACA)',
    width=600,
    height=450,
    template="simple_white",
    plot_bgcolor="white",
    margin=go.layout.Margin(l=0, r=20, b=0, t=50),
    legend_font_size=14.5,
    xaxis_title='',
    yaxis_title='Energy Requirement (GJ/tonne)',
    barmode='stack',
    showlegend = True
)

# Show plot
fig.show()

fig.write_image(outputDirectory_general + date + "_DiscussionDv1_IsoBut_Gly" + ".png", scale=5)

### Bio Scatterplot ###

In [ ]:
scenario = 'Categories' #Ideal or Realistic
case = 'Category_2' #Worst, Base, Best
FileName = '2024-05-08_Type-Bio_Case-'+ case +'_Analysis-' + scenario + '_FULL.xlsx'
SheetName = 'Overall-ECR-Bio'
Type = 'ECR-Bio'

df = readInData(FileName, SheetName, Type)
n = 57

df.loc[df['SubstratePathway'] == 'FOR_1:H2_0', 'SubstrateType'] = 'C1+H2'
df.loc[df['SubstratePathway'] == 'CO_1:H2_0', 'SubstrateType'] = 'C1+H2'


#To graph only a subset of bioproducts:
#df = df[(df['BioProduct'].isin(['Propane-PW1','Butanol-PW2', '1-3-Diaminopropane', 'Isoprene', '2-3-Butanediol','Lysine_L', 'Catechol', 'Adipate', 'Succinate', 'Malonate']))]
#n = 10

display(df)

In [13]:
##----- Bioproduction Yields Scatterplot -----##

#df = df.loc[df['MoleculeType_P'] == 'Hydrocarbon (alkane)']
#df = df.sort_values(by='C_P', ascending=0)
df = df.sort_values(by='Deg_Red_P', ascending=0)
#display(df)

ORI = 'h' #'v' for vertical, 'h' for horizontal

if ORI is 'v':
    X = 'ProductName_Graph'
    Y = 'Yield_Mass'
    W = 1400
    H = 800
    title_pos = 0.45
    
elif ORI is 'h':
    X = 'Yield_Mass'
    Y = 'ProductName_Graph'
    W = 1200
    H = 1500
    title_pos = 0.6


markerSize=7

fig = go.Figure()


dfS1 = df.loc[df['Substrate'] == 'Glucose'] 
dataS1 = go.Scatter(x=dfS1[X], y=dfS1[Y], mode='markers', orientation=ORI,
                    marker_color= 'red', marker_symbol= 'diamond', marker_size=markerSize,
                    name = 'Glucose'
                    )

dfS2 = df.loc[df['Substrate'] == 'Glycerol']
dataS2 = go.Scatter(x=dfS2[X], y=dfS2[Y], mode='markers', orientation=ORI,
                    marker_color= 'gold',marker_symbol= 'diamond', marker_size=markerSize,
                    name = 'Glycerol'
                    )

dfS3 = df.loc[df['Substrate'] == 'Xylose'] 
dataS3 = go.Scatter(x=dfS3[X], y=dfS3[Y], mode='markers', orientation=ORI,
                    marker_color= 'blue', marker_symbol= 'diamond', marker_size=markerSize,
                    name = 'Xylose'
                    )

#ECR 1-C compounds; iJO1366
dfC11 = df.loc[df['Substrate'] == 'Methanol_Formolase']
dataC11 = go.Scatter(x=dfC11[X], y=dfC11[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkviolet', marker_symbol= 'square', marker_size = markerSize+1,
                    name = 'MeOH - Formolase',    
                    )

dfC12 = df.loc[df['Substrate'] == 'Methanol_RuMP']
dataC12 = go.Scatter(x=dfC12[X], y=dfC12[Y], mode='markers', orientation=ORI,
                    marker_color= 'deepskyblue', marker_symbol= 'square', marker_size = markerSize-1,
                    name = 'MeOH - RuMP',
                    )

dfC13 = df.loc[df['Substrate'] == 'Methanol_SerineCycle']
dataC13 = go.Scatter(x=dfC13[X], y=dfC13[Y], mode='markers', orientation=ORI,
                    marker_color= 'red', marker_symbol= 'square', marker_size = markerSize,
                    name = 'MeOH - SerineCycle'    
                    )

dfC14 = df.loc[df['Substrate'] == 'Formate_Formolase']
dataC14 = go.Scatter(x=dfC14[X], y=dfC14[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkviolet', marker_symbol= 'x', marker_size = markerSize,
                    name = 'Formate - Formolase',
                    )

dfC15 = df.loc[df['Substrate'] == 'Formate_RGP-Serine']
dataC15 = go.Scatter(x=dfC15[X], y=dfC15[Y],mode='markers', orientation=ORI,
                    marker_color= 'olivedrab', marker_symbol= 'x', marker_size = markerSize,
                    name = 'Formate - RGP',
                    )

dfC16 = df.loc[df['Substrate'] == 'Formate_SerineCycle']
dataC16 = go.Scatter(x=dfC16[X], y=dfC16[Y],mode='markers', orientation=ORI,
                    marker_color= 'red', marker_symbol= 'x', marker_size = markerSize,
                    name = 'Formate - Serine Cycle',
                    )

#ECR 2-C compounds; iJO1366
dfC21 = df.loc[df['Substrate'] == 'Acetaldehyde']
dataC21 = go.Scatter(x=dfC21[X], y=dfC21[Y], mode='markers', orientation=ORI,
                    marker_color='darkblue', marker_symbol= 'circle', marker_size =markerSize,
                    name = 'Acetaldehyde',
                    )

dfC22 = df.loc[df['Substrate'] == 'Acetate']
dataC22 = go.Scatter(x=dfC22[X], y=dfC22[Y], mode='markers', orientation=ORI,
                    marker_color= 'red', marker_symbol= 'circle', marker_size =markerSize,
                    name = 'Acetate',
                    )


dfC23 = df.loc[df['Substrate'] == 'Ethanol']
dataC23 = go.Scatter(x=dfC23[X], y=dfC23[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkorange',marker_symbol= 'circle', marker_size =markerSize,
                    name = 'Ethanol',
                    )

dfC24 = df.loc[df['Substrate'] == 'EG_Glycolate']
dataC24 = go.Scatter(x=dfC24[X], y=dfC24[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkblue', marker_symbol= 'triangle-up', marker_size =markerSize,
                    name = 'EG - Glycolate',
                    )

dfC25 = df.loc[df['Substrate'] == 'EG_SACA']
dataC25 = go.Scatter(x=dfC25[X], y=dfC25[Y],mode='markers', orientation=ORI,
                    marker_color= 'red', marker_symbol= 'triangle-up', marker_size =markerSize,
                    name = 'EG - SACA',
                    )



dfC26 = df.loc[df['Substrate'] == 'Glycolaldehyde_Glycolate']
dataC26 = go.Scatter(x=dfC26[X], y=dfC26[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkorange', marker_symbol= 'triangle-up', marker_size =markerSize,
                    name = 'GlyAld - Glycolate',
                    )

dfC27 = df.loc[df['Substrate'] == 'Glycolaldehyde_SACA']
dataC27 = go.Scatter(x=dfC27[X], y=dfC27[Y], mode='markers', orientation=ORI,
                    marker_color= 'deepskyblue', marker_symbol= 'triangle-up', marker_size =markerSize,
                    name = 'GlyAld - SACA',
                    )


#ECR - iHN637 - CO + Formate Mixes
dfH1 = df.loc[df['Substrate'] == 'FOR_1:H2_0'] 
dataH1 = go.Scatter(x=dfH1[X], y=dfH1[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkgreen', marker_symbol= 'star-square-open', marker_size =markerSize,
                    name = '1 FOR:0 H2',
                    )

dfH2 = df.loc[df['Substrate'] == 'FOR_1:H2_0.5']
dataH2 = go.Scatter(x=dfH2[X], y=dfH2[Y], mode='markers', orientation=ORI,
                    marker_color= 'red', marker_symbol= 'star-square-open', marker_size =markerSize,
                    name = '1 FOR:0.5 H2',
                    )

dfH3 = df.loc[df['Substrate'] == 'FOR_1:H2_1']
dataH3 = go.Scatter(x=dfH3[X], y=dfH3[Y], mode='markers', orientation=ORI,
                    marker_color= 'dodgerblue', marker_symbol= 'star-square-open', marker_size =markerSize,
                    name = '1 FOR:1 H2',
                    )

dfH4 = df.loc[df['Substrate'] == 'FOR_1:H2_2']
dataH4 = go.Scatter(x=dfH4[X], y=dfH4[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkviolet', marker_symbol= 'star-square-open', marker_size =markerSize,
                    name = '1 FOR:2 H2',
                    )

dfH5 = df.loc[df['Substrate'] == 'CO_1:H2_0']
dataH5 = go.Scatter(x=dfH5[X], y=dfH5[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkgreen', marker_symbol= 'diamond-tall-open', marker_size =markerSize+0.5,
                    name = '1 CO:0 H2',
                    )

dfH6 = df.loc[df['Substrate'] == 'CO_1:H2_0.5'] 
dataH6 = go.Scatter(x=dfH6[X], y=dfH6[Y], mode='markers', orientation=ORI,
                    marker_color= 'red', marker_symbol= 'diamond-tall-open', marker_size =markerSize+0.5,
                    name = '1 CO:0.5 H2',
                    )

dfH7 = df.loc[df['Substrate'] == 'CO_1:H2_1']
dataH7 = go.Scatter(x=dfH7[X], y=dfH7[Y], mode='markers', orientation=ORI,
                    marker_color= 'dodgerblue', marker_symbol= 'diamond-tall-open', marker_size =markerSize+0.5,
                    name = '1 CO:1 H2',
                    )

dfH8 = df.loc[df['Substrate'] == 'CO_1:H2_2']
dataH8 = go.Scatter(x=dfH8[X], y=dfH8[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkviolet', marker_symbol= 'diamond-tall-open',  marker_size =markerSize+0.5,
                    name = '1 CO:2 H2',
                    )

dfH9 = df.loc[df['Substrate'] == 'CO_2:H2_2']
dataH9 = go.Scatter(x=dfH9[X], y=dfH9[Y], mode='markers', orientation=ORI,
                    marker_color= 'black', marker_symbol= 'circle-open',  marker_size =markerSize,
                    name = '1 CO2:X H2',
                    )

#ECR - iHN637 - Methanogensis

dfH10 = df.loc[df['Substrate'] == 'Methanol_MethylTransferase_1']
dataH10 = go.Scatter(x=dfH10[X], y=dfH10[Y], mode='markers', orientation=ORI,
                    marker_color= 'dodgerblue', marker_symbol= 'star-open', marker_size = markerSize+4,
                    name = 'Methanol Fixation 1 ATP',    
                    )

dfH11 = df.loc[df['Substrate'] == 'Methanol_MethylTransferase_0.5']
dataH11 = go.Scatter(x=dfH11[X], y=dfH11[Y], mode='markers', orientation=ORI,
                    marker_color= 'red', marker_symbol= 'star-open', marker_size = markerSize+3,
                    name = 'Methanol Fixation 0.5 ATP',    
                    )

dfH12 = df.loc[df['Substrate'] == 'Methanol_MethylTransferase_0.1']
dataH12 = go.Scatter(x=dfH12[X], y=dfH12[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkgreen', marker_symbol= 'star-open', marker_size = markerSize+2,
                    name = 'Methanol Fixation 0.1 ATP',    
                    )

dfH13 = df.loc[df['Substrate'] == 'Methanol_MethylTransferase_0']
dataH13 = go.Scatter(x=dfH13[X], y=dfH13[Y], mode='markers', orientation=ORI,
                    marker_color= 'darkviolet', marker_symbol= 'star-open', marker_size = markerSize+1,
                    name = 'Methanol Fixation no ATP',    
                    )
                    

data = [dataC21, dataC22, dataC23, dataC24, dataC25, dataC26, dataC27,
        dataC11, dataC12, dataC13, dataC14, dataC15, dataC16,
        dataH1, dataH2, dataH3, dataH4, dataH5, dataH6, dataH7, dataH8, dataH9,
        dataS1, dataS2, dataS3, dataH10, dataH11, dataH12, dataH13
        ]

if ORI is 'v':
    shape1=dict(type="line", x0=-1, x1=57, y0=1, y1=1, xref='x', yref='y', line=dict(color='black', width=1.5))
elif ORI is 'h':
    shape1=dict(type="line", x0=1, x1=1, y0=-1, y1=57, xref='x', yref='y', line=dict(color='black', width=1.5))
#shape2=dict(type="rect", x0=6, x1=10, y0=0, y1=2.5, xref='x', yref='y',fillcolor='orange', opacity=0.2, line=dict(color='darkorange', width=2))


shapes = [shape1]

layout = go.Layout(title_text='Bioproduction Mass Yields', title = {'x': title_pos},
                   width=W, height=H, template="simple_white", plot_bgcolor = "white",
                   margin=go.layout.Margin(l=0, r=20,b=0, t=50),
                  legend_font_size=14.5, #20 #for ORI=h,#)
                  shapes=shapes)

fig = go.Figure(data = data, layout = layout)



if ORI == 'v':
        fig.update_xaxes(title_text = 'Bioproduct', title_font_size= 16, tickfont_size=14, tickangle=-90,
                 showline=True, linewidth=1, color='black', mirror = True,
                 showgrid=True, gridcolor='#eee',
                 range=[-1,57])
        fig.update_yaxes(title_text = 'Mass Yield (g/g)', title_font_size= 16, tickfont_size=14,
                 showline=True, linewidth=1, color='black', mirror = True, showgrid=True, gridcolor='#eee',
                 range=[0,2.5])
        
    
    
elif ORI == 'h':

    fig.update_xaxes(title_text = 'Mass Yield (g/g)', title_font_size= 20, tickfont_size=20,
                 showline=True, linewidth=1, color='black', mirror = True, showgrid=True, gridcolor='#eee',
                range=[0,2.5], tick0 = 0, dtick= 0.2)
    fig.update_yaxes(title_text = 'Bioproducts', title_font_size= 20, tickfont_size=15,
                 showline=True, linewidth=1, color='black', mirror = True,
                 showgrid=True, gridcolor='#eee',
                 range=[-1,57]) 

    fig.update_layout(legend=dict(yanchor="bottom", y=0.01, xanchor="right", x=0.99))

fig.show()
fig.write_image(outputDirectory_general + date + "_BioproductionYields_Scatter_" + ORI + ".png", scale=5)

<>:10: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:17: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:220: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:222: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:10: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:17: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:220: SyntaxWarning:

"is" with a literal. Did you mean "=="?

<>:222: SyntaxWarning:

"is" with a literal. Did you mean "=="?

C:\Users\austi\AppData\Local\Temp\ipykernel_5136\1793348294.py:10: SyntaxWarning:

"is" with a literal. Did you mean "=="?

C:\Users\austi\AppData\Local\Temp\ipykernel_5136\1793348294.py:17: SyntaxWarning:

"is" with a literal. Did you mean "=="?

C:\Users\austi\AppData\Local\Temp\ipykernel_5136\1793348294.py:220: SyntaxWarning:

"is" with a literal. Did you mean "=="?

C:\Users\austi\AppData\Local\Temp\ipykernel_5136\1793348294.py:222: SyntaxWarning:

"is" with a literal. Did yo